# SurfaceGuard AI — KAGGLE V19: Research-Guided, Audit-Gated Eight-Model Hybrid

**Purpose:** a controlled next experiment against the user's confirmed Kaggle public score of **0.64701** (V9.1, Version 8). The notebook does not promise a particular leaderboard gain. It is designed to reject fragile calibration changes rather than maximize one validation number.

## Eight existing checkpoints (inference only)

**Base segmentation models**
- V4: U-Net / ResNet34
- V5: SegFormer-B0
- V6: DeepLabV3+ / ResNet50
- MiT-B4: SegFormer-B4

**Auxiliary segmentation + image-presence models**
- Colab V7 A: scratch ResNet34 U-Net + classifier
- Colab V7 B: pretrained ResNet34 U-Net + classifier
- Colab V7 C: SegFormer MiT-B0 + pooled-logit classifier
- Colab V7 D: DeepLabV3+ / ResNet50 + classifier

Attach the competition dataset, `shrutikarthikeyan/v4-v5-v6-mit`, and the Kaggle dataset `shrutikarthikeyan/surfaceguard-colab-v7-checkpoints`. All eight checkpoints are required. No model is retrained.

## What V19 changes

1. Retains V19's class-aware four-model blend, auxiliary segmentation fusion, probability/logit fusion, adaptive classifier thresholding, optional pixel-conflict resolution, and four-way flip TTA.
2. Splits the original internal calibration partition into:
   - **CAL_TUNE:** chooses pixel thresholds, blend weights, beta values, and classifier gates.
   - **CAL_AUDIT:** independently decides whether the selected soft-gate and pixel-conflict policies are robust enough to keep.
   - **HOLDOUT:** remains diagnostic only and is never used to select a policy.
3. Uses a minimum positive-pair support rule before accepting a soft-gate change for a rare class. If the audit subset has too few positive examples, that class keeps the conservative hard-gate policy.
4. Rejects a soft-gate or conflict-resolution change when its CAL_AUDIT gain is below a small predeclared margin. This is intended to reduce calibration overfit, not guarantee a public-LB increase.
5. Keeps component filtering disabled by default. Prior project experiments found it could improve calibration while reducing reserved-holdout performance.
6. Keeps pseudo-labelling disabled. Related competition write-ups report mixed results, including private-LB degradation from pseudo-labeling.

## Research synthesis — related work, not direct score transfer

The closest public analogues are steel-defect segmentation competitions, not this exact 128×800 competition:

- The Severstal 3rd-place write-up used a diverse nine-model ensemble spanning EfficientNet-B3/B4/B5 and SE-ResNeXt50 with FPN/U-Net, plus class-specific label/pixel thresholds. Its strongest transferable lesson is **model diversity**, not the numeric score. [Kaggle 3rd-place write-up](https://www.kaggle.com/competitions/severstal-steel-defect-detection/writeups/chienyichi-3rd-place-solution)
- The 12th-place Severstal solution used a classifier → multi-label segmentation → binary segmentation cascade, crop-based training, and carefully calibrated small-object filtering. This supports keeping image-level class-presence signals, but filtering must be validated on this dataset. [Kaggle 12th-place write-up](https://www.kaggle.com/competitions/severstal-steel-defect-detection/writeups/ods-ai-dip-12th-place-solution-overview)
- A 40th-place write-up explicitly cited low thresholds, pseudo-labelling, and weak validation as possible reasons for private-LB decline. V19 therefore avoids pseudo-labelling and introduces a separate audit subset. [Kaggle 40th-place write-up](https://www.kaggle.com/competitions/severstal-steel-defect-detection/writeups/knowledge-disteelation-small-writeup-from-20th-to-)
- A GitHub solution report shows a public/private leaderboard gap after choosing a public-LB-optimised submission. It also reports different public/private behaviour for RMS versus mean probability averaging. V19 does not switch to RMS pooling. [Khornlund's report](https://github.com/khornlund/severstal-steel-defect-detection)
- A Reddit discussion on thin scratches on reflective metal highlights low-contrast defects, reflection-driven false positives, hard-negative mining, and confidence-aware filtering. It is a small, separate dataset and is treated as anecdotal inspiration only. [Reddit discussion](https://www.reddit.com/r/computervision/comments/1rxajdc/detecting_thin_scratches_on_reflective_metal/)
- Recent steel-defect reviews emphasize multi-scale feature fusion and boundary/detail preservation, while warning that attention/transformer modules can add complexity and do not automatically generalize. [2026 review](https://www.mdpi.com/2079-6412/16/9/1040) · [industrial inspection review](https://www.mdpi.com/2227-7080/14/10/644)

**Most promising future capacity experiment:** train a genuinely different FPN model with an SE-ResNeXt50 or EfficientNet encoder and test it as an independent ninth model. The current V19 is deliberately inference-only; it does not pretend that adding more calibration knobs is equivalent to adding a new source of learned features.

## Official metric used by this notebook

\[
Score=0.70\,ForegroundDice+0.20\,FalsePositiveScore+0.10\,EmptyAccuracy
\]

All search/evaluation functions use the competition's pooled global denominators. Per-class contributions are additive components of the same global score; they are not four independent class-macro scores.

## Runtime and submission

- GPU inference uses four-way horizontal/vertical flip TTA; CPU fallback uses one pass and is not directly comparable to GPU/TTA validation.
- CUDA OOM recovery switches all loaded models and data loaders to CPU.
- The notebook writes `/kaggle/working/submission.csv` and a separate V19 backup.
- Empty masks are encoded as exactly one space (`" "`); missing predictions and NaN values are hard errors.
- Save V19 and all earlier submissions. Only the Kaggle public score can confirm whether V19 improves on 0.64701.


In [ ]:
# Install SMP without resolving/replacing Kaggle's preinstalled PyTorch/CUDA stack.
# Kaggle GPU images already provide torch/torchvision; resolving dependencies can
# otherwise replace that build with a CPU wheel.
%pip install -q --disable-pip-version-check --no-deps segmentation-models-pytorch==0.6.0

from importlib.metadata import version
import torch

smp_version = version('segmentation-models-pytorch')
print('Installed SMP version:', smp_version)
if smp_version != '0.6.0':
    print(f'NOTE: SMP version is {smp_version}; continuing because the checkpoint architecture is loaded strictly below.')
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

import segmentation_models_pytorch as smp
print('SMP import successful.')

if torch.cuda.is_available():
    try:
        print('GPU:', torch.cuda.get_device_name(0))
    except Exception as exc:
        print('GPU probe warning:', type(exc).__name__, str(exc)[:200])
else:
    print('GPU unavailable at startup; the notebook will use CPU mode automatically.')

In [ ]:
import os
import copy
import gc
import json
import time
import math
import shutil
import zipfile
import hashlib
import warnings
from pathlib import Path, PurePosixPath
from contextlib import nullcontext

import numpy as np
import pandas as pd
from scipy import ndimage
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import segmentation_models_pytorch as smp
from transformers import SegformerConfig, SegformerForSemanticSegmentation

SEED = 42
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
HEIGHT, WIDTH = 128, 800
N_CLASSES = len(CLASSES)
N_PIXELS = HEIGHT * WIDTH
BEST_PUBLIC_SCORE_BASELINE = 0.64701  # User-confirmed Kaggle public score: V9.1, Version 8
LOCAL_VALIDATION_TARGET = 0.675
PUBLIC_SCORE_TARGET = 0.68

random_state = np.random.RandomState(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Select CUDA only when it is genuinely usable and has enough free memory for
# four loaded segmentation models plus inference activations. Otherwise use CPU.
MIN_FREE_GPU_GB = 6.0
DEVICE = torch.device("cpu")
GPU_NAME = "CPU fallback"
GPU_FREE_GB = 0.0
GPU_TOTAL_GB = 0.0

if torch.cuda.is_available():
    try:
        torch.cuda.init()
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        GPU_FREE_GB = free_bytes / (1024 ** 3)
        GPU_TOTAL_GB = total_bytes / (1024 ** 3)
        if GPU_FREE_GB >= MIN_FREE_GPU_GB:
            _probe = torch.empty((1,), device="cuda:0")
            torch.cuda.synchronize()
            del _probe
            DEVICE = torch.device("cuda:0")
            GPU_NAME = torch.cuda.get_device_name(0)
        else:
            print(
                f"CUDA detected but only {GPU_FREE_GB:.2f} GiB is free; "
                f"minimum is {MIN_FREE_GPU_GB:.1f} GiB. Selecting CPU fallback."
            )
    except Exception as exc:
        DEVICE = torch.device("cpu")
        GPU_NAME = "CPU fallback (CUDA initialisation failed)"
        print("CUDA initialisation failed; selecting CPU fallback:", type(exc).__name__, str(exc)[:250])

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    # Avoid cuDNN's benchmark workspace spikes; fixed input dimensions do not
    # require benchmark mode for reliable inference.
    torch.backends.cudnn.benchmark = False
    MAIN_BATCH_SIZE = 1
    MIT_BATCH_SIZE = 1
    TEST_BATCH_SIZE = 1
    NUM_WORKERS = 2
else:
    # CPU mode uses single-image batches and no worker processes for reliability.
    MAIN_BATCH_SIZE = 1
    MIT_BATCH_SIZE = 1
    TEST_BATCH_SIZE = 1
    NUM_WORKERS = 0
    try:
        torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass  # PyTorch may already have initialised its thread pools.


def _is_cuda_oom_error(exc):
    oom_type = getattr(torch.cuda, "OutOfMemoryError", None)
    if oom_type is not None and isinstance(exc, oom_type):
        return True
    message = str(exc).lower()
    return ("out of memory" in message and ("cuda" in message or "gpu" in message)) or "cuda error: out of memory" in message


def _rebuild_loaders_for_cpu():
    """Switch existing, already-created DataLoaders to safe CPU settings."""
    global val_loader_main, val_loader_mit, test_loader_main, test_loader_mit
    global MAIN_BATCH_SIZE, MIT_BATCH_SIZE, TEST_BATCH_SIZE, NUM_WORKERS
    MAIN_BATCH_SIZE = MIT_BATCH_SIZE = TEST_BATCH_SIZE = 1
    NUM_WORKERS = 0
    loader_specs = [
        ("val_dataset_main", "val_loader_main", MAIN_BATCH_SIZE),
        ("val_dataset_mit", "val_loader_mit", MIT_BATCH_SIZE),
        ("test_dataset_main", "test_loader_main", TEST_BATCH_SIZE),
        ("test_dataset_mit", "test_loader_mit", TEST_BATCH_SIZE),
    ]
    for dataset_name, loader_name, batch_size in loader_specs:
        dataset = globals().get(dataset_name)
        if dataset is not None:
            globals()[loader_name] = DataLoader(
                dataset, batch_size=batch_size, shuffle=False,
                num_workers=0, pin_memory=False, drop_last=False
            )


def switch_to_cpu_fallback(reason, extra_models=()):
    """Move every loaded model to CPU after a CUDA OOM; return whether switched."""
    global DEVICE, GPU_NAME
    if DEVICE.type != "cuda":
        return False
    print("\nCUDA memory/runtime allocation failed. Switching all models to CPU.")
    print("Reason:", str(reason)[:300])
    candidates = list(extra_models)
    for model_name in (
        "model_v4", "model_v5", "model_v6", "model_mit",
        "model_colab_a", "model_colab_b", "model_colab_c", "model_colab_d"
    ):
        candidate = globals().get(model_name)
        if isinstance(candidate, torch.nn.Module):
            candidates.append(candidate)
    seen = set()
    for model in candidates:
        if model is None or id(model) in seen:
            continue
        seen.add(id(model))
        try:
            model.to(torch.device("cpu"))
            model.eval()
        except Exception as move_exc:
            raise RuntimeError(
                "CUDA fallback was triggered, but a loaded model could not be moved to CPU."
            ) from move_exc
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    DEVICE = torch.device("cpu")
    GPU_NAME = "CPU fallback (automatic CUDA OOM recovery)"
    try:
        torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass
    _rebuild_loaders_for_cpu()
    print("Fallback complete. All available model objects are on CPU.")
    return True

OUTPUT_DIR = Path("/kaggle/working/surfaceguard_version19_audit_gated_hybrid")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Validation prediction caches are temporary disk-backed arrays. They are not loaded
# fully into RAM and are not retained as large notebook output artifacts.
CACHE_DIR = Path("/kaggle/temp/surfaceguard_version19_audit_gated_hybrid_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("SurfaceGuard V19 score-optimisation inference pipeline")
print("GPU:", GPU_NAME)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("SMP:", smp.__version__)
print("Classes:", CLASSES)
print("Image size:", (HEIGHT, WIDTH))
print("Local validation reference:", LOCAL_VALIDATION_TARGET)
print("Current best public baseline:", BEST_PUBLIC_SCORE_BASELINE)
print("Public leaderboard goal:", PUBLIC_SCORE_TARGET)
print("Execution device:", DEVICE, "|", GPU_NAME)
if DEVICE.type == "cuda":
    print(f"GPU memory at startup: {GPU_FREE_GB:.2f}/{GPU_TOTAL_GB:.2f} GiB free/total")
    print("GPU mode: mixed-precision inference with conservative batch size 1.")
else:
    print("CPU fallback mode: safe batch size 1, zero DataLoader workers; TTA is reduced to one pass to keep runtime manageable.")
    print("NOTE: for the full V19 four-way TTA candidate, enable a Kaggle GPU accelerator before running all cells.")
print("Output directory:", OUTPUT_DIR)
print("Optional Colab V7 auxiliary models will be included only when all four checkpoint files are attached.")
print("No model-training code will be executed in this notebook. V19 changes inference-time fusion/calibration only.")


In [ ]:
INPUT_ROOT = Path("/kaggle/input")
MODEL_CACHE_DIR = Path("/kaggle/temp/surfaceguard_model_inputs")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_ROOT.exists():
    raise FileNotFoundError("/kaggle/input does not exist. Attach the required Kaggle inputs.")

REQUIRED_ARTIFACTS = {
    "best_v6_deeplabv3plus.pt",
    "best_mit_b4.pt",
    "mit_b4_holdout_report.json",
    "best_v4.pt",
    "best_v5.pt",
    # Optional Colab V7 auxiliary-classifier models. Extracted only when present.
    "model_a_scratch.pt",
    "model_b_pretrained.pt",
    "model_c_segformer.pt",
    "model_d_deeplabv3plus.pt",
}
ARCHIVE_MARKERS = {
    "data.pkl", "version", "byteorder", ".format_version", ".storage_alignment"
}


def _safe_zip_parts(member_name):
    p = PurePosixPath(member_name)
    if p.is_absolute() or ".." in p.parts:
        return None
    return p.parts


def _copy_zip_member(zf, member, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zf.open(member, "r") as source, open(destination, "wb") as target:
        shutil.copyfileobj(source, target)


# Extract direct model/report files and split V4/V5 torch archives from attached ZIPs.
zip_extractions = []
for zip_path in sorted(INPUT_ROOT.rglob("*.zip")):
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zip_identity = hashlib.sha256(
                str(zip_path.resolve()).encode("utf-8")
            ).hexdigest()[:12]
            stage_root = (
                MODEL_CACHE_DIR / "zip_extracted" / f"{zip_path.stem}_{zip_identity}"
            )
            for info in zf.infolist():
                if info.is_dir():
                    continue
                parts = _safe_zip_parts(info.filename)
                if not parts:
                    continue
                basename = parts[-1]

                if basename in REQUIRED_ARTIFACTS:
                    # Preserve the member path: two different artifacts with the same
                    # basename must be surfaced as duplicates, never silently ignored.
                    target = stage_root / "artifacts" / Path(*parts)
                    if not target.exists():
                        _copy_zip_member(zf, info, target)
                        zip_extractions.append(str(target))
                    continue

                # A PyTorch split archive has files such as best_v4/data.pkl,
                # best_v4/version, best_v4/data/0, best_v4/.data/serialization_id.
                for model_name in ("best_v4", "best_v5"):
                    matching = [i for i, part in enumerate(parts) if part == model_name]
                    if not matching:
                        continue
                    root_index = matching[-1]  # use the innermost best_v4/best_v5 folder
                    relative_parts = parts[root_index:]
                    if len(relative_parts) < 2:
                        break
                    relative_path = PurePosixPath(*relative_parts)
                    target = stage_root / "archives" / Path(*relative_path.parts)
                    if not target.exists():
                        _copy_zip_member(zf, info, target)
                        zip_extractions.append(str(target))
                    break
    except zipfile.BadZipFile:
        print("Skipping non-ZIP or damaged ZIP:", zip_path)

print("Extracted checkpoint artifacts:", len(zip_extractions))


def _sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_unique_file(filename, required=True):
    hits = []
    for root in (INPUT_ROOT, MODEL_CACHE_DIR):
        if root.exists():
            hits.extend(p for p in root.rglob(filename) if p.is_file())
    hits = list({str(p.resolve()): p for p in hits}.values())
    if not hits:
        if required:
            raise FileNotFoundError(
                f"Required artifact '{filename}' was not found under /kaggle/input. "
                "Attach the correct Notebook Output Files or a ZIP containing this exact file."
            )
        return None
    if len(hits) == 1:
        return hits[0]

    sizes = {p.stat().st_size for p in hits}
    if len(sizes) == 1:
        hashes = {_sha256(p) for p in hits}
        if len(hashes) == 1:
            return sorted(hits, key=lambda p: (len(str(p)), str(p)))[0]

    listing = "\n".join(f"{p} ({p.stat().st_size:,} bytes)" for p in hits)
    raise RuntimeError(
        f"Multiple different files named {filename} were found. Keep only the intended "
        f"checkpoint version attached. Candidates:\n{listing}"
    )


def reconstruct_v4_v5_archive(model_name):
    """Return direct .pt checkpoint or reconstruct it from a split PyTorch archive folder."""
    direct = find_unique_file(f"{model_name}.pt", required=False)
    if direct is not None:
        return direct

    roots = []
    for search_root in (INPUT_ROOT, MODEL_CACHE_DIR):
        if search_root.exists():
            for data_pickle in search_root.rglob("data.pkl"):
                archive_root = data_pickle.parent
                if archive_root.name == model_name and (archive_root / "version").is_file():
                    roots.append(archive_root)
    roots = list({str(p.resolve()): p for p in roots}.values())

    if not roots:
        raise FileNotFoundError(
            f"Could not find {model_name}.pt or its split archive folder. "
            "Attach surfaceguard-v4-v5-checkpoints or a ZIP containing "
            f"{model_name}/data.pkl, {model_name}/version and {model_name}/data/*."
        )

    if len(roots) > 1:
        preferred = [p for p in roots if "surfaceguard-v4-v5-checkpoints" in str(p)]
        if len(preferred) == 1:
            source = preferred[0]
        else:
            raise RuntimeError(
                f"Multiple candidate split archives were found for {model_name}: {roots}. "
                "Keep only the intended checkpoint dataset/ZIP attached."
            )
    else:
        source = roots[0]

    output_path = MODEL_CACHE_DIR / f"{model_name}.pt"
    with zipfile.ZipFile(output_path, "w", compression=zipfile.ZIP_STORED) as zf:
        for file_path in source.rglob("*"):
            if file_path.is_file():
                archive_name = (Path(model_name) / file_path.relative_to(source)).as_posix()
                zf.write(file_path, archive_name)

    if not zipfile.is_zipfile(output_path):
        raise RuntimeError(f"Reconstructed checkpoint is not a valid ZIP archive: {output_path}")
    return output_path


V4_PATH = reconstruct_v4_v5_archive("best_v4")
V5_PATH = reconstruct_v4_v5_archive("best_v5")
V6_PATH = find_unique_file("best_v6_deeplabv3plus.pt")
MIT_PATH = find_unique_file("best_mit_b4.pt")
MIT_REPORT_PATH = find_unique_file("mit_b4_holdout_report.json")


# V19: all four Colab V7 auxiliary checkpoints are REQUIRED for this hybrid run.
# Kaggle cannot read files directly from Google Drive. Create a Kaggle Dataset
# containing the four .pt files, attach it to this notebook, and rerun from the top.
COLAB_V7_FILENAMES = {
    "a_scratch": "model_a_scratch.pt",
    "b_pretrained": "model_b_pretrained.pt",
    "c_segformer_b0": "model_c_segformer.pt",
    "d_deeplabv3plus": "model_d_deeplabv3plus.pt",
}
COLAB_V7_PATHS = {
    key: find_unique_file(filename, required=False)
    for key, filename in COLAB_V7_FILENAMES.items()
}
COLAB_V7_MISSING = [
    COLAB_V7_FILENAMES[key] for key, path in COLAB_V7_PATHS.items() if path is None
]
COLAB_V7_AVAILABLE = len(COLAB_V7_MISSING) == 0

print("Colab V7 auxiliary checkpoint preflight")
for key, filename in COLAB_V7_FILENAMES.items():
    path = COLAB_V7_PATHS[key]
    if path is None:
        print(f"  MISSING: {filename}")
    else:
        print(f"  FOUND:   {filename} | {path} | {path.stat().st_size / (1024**2):.2f} MB")

if COLAB_V7_MISSING:
    discovered = sorted({p.name for p in INPUT_ROOT.rglob("*.pt")}) if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        "V19 requires all four Colab V7 auxiliary checkpoints for the eight-model hybrid.\\n"
        f"Missing files: {COLAB_V7_MISSING}\\n"
        "Kaggle cannot access your Google Drive directly. Create a Kaggle Dataset containing "
        "model_a_scratch.pt, model_b_pretrained.pt, model_c_segformer.pt and "
        "model_d_deeplabv3plus.pt; attach it to this notebook; then choose Save Version -> Save & Run All.\\n"
        f".pt files currently discovered under /kaggle/input: {discovered}"
    )

print("All four Colab V7 checkpoints found. Eight-model hybrid is enabled.")

preferred_data_root = Path(
    "/kaggle/input/competitions/industrial-surface-defect-detection/public"
)
if (preferred_data_root / "train.csv").is_file():
    DATA_ROOT = preferred_data_root
else:
    candidates = []
    for train_csv in INPUT_ROOT.rglob("train.csv"):
        parent = train_csv.parent
        if (parent / "train").is_dir() and (parent / "test").is_dir() and (parent / "sample_submission.csv").is_file():
            candidates.append(parent)
    candidates = list({str(p.resolve()): p for p in candidates}.values())
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Could not uniquely locate competition data. Attach industrial-surface-defect-detection. "
            f"Candidates found: {candidates}"
        )
    DATA_ROOT = candidates[0]

print("\nINPUT PREFLIGHT PASSED")
for label, path in [
    ("V4", V4_PATH), ("V5", V5_PATH), ("V6", V6_PATH),
    ("MiT-B4", MIT_PATH), ("MiT report", MIT_REPORT_PATH),
    ("Competition data", DATA_ROOT)
]:
    print(f"{label:16s}: {path}")
    if Path(path).is_file():
        print(" " * 18, f"{Path(path).stat().st_size / (1024**2):.2f} MB")


In [ ]:
train_df = pd.read_csv(
    DATA_ROOT / "train.csv",
    dtype={"image_id": str, "defect_type": str}
)

required_columns = {"image_id", "defect_type", "mask_rle"}
if not required_columns.issubset(train_df.columns):
    raise ValueError(f"train.csv is missing required columns: {required_columns - set(train_df.columns)}")

train_df["image_id"] = train_df["image_id"].astype(str)
train_df["defect_type"] = train_df["defect_type"].astype(str)
train_df["mask_rle"] = train_df["mask_rle"].fillna("").astype(str)
train_df["has_defect"] = train_df["mask_rle"].str.strip().ne("")

presence = train_df.pivot_table(
    index="image_id",
    columns="defect_type",
    values="has_defect",
    aggfunc="max",
    fill_value=False
).reindex(columns=CLASSES, fill_value=False).astype(bool)

all_image_ids = presence.index.astype(str).to_numpy()
label_combo = presence.astype(int).astype(str).agg("".join, axis=1)

train_ids, val_ids = train_test_split(
    all_image_ids,
    test_size=0.20,
    random_state=SEED,
    stratify=label_combo.loc[all_image_ids]
)
train_ids = np.asarray(train_ids, dtype=str)
val_ids = np.asarray(val_ids, dtype=str)

if len(all_image_ids) != 11393 or len(train_ids) != 9114 or len(val_ids) != 2279:
    raise RuntimeError(
        f"Unexpected data split: total={len(all_image_ids)}, train={len(train_ids)}, val={len(val_ids)}. "
        "This is not the expected seed-42 80/20 split used by the saved models."
    )
if set(train_ids).intersection(set(val_ids)):
    raise RuntimeError("Training/validation image overlap detected.")

# Recreate the MiT-B4 V2 internal calibration/holdout split.
val_combo = label_combo.loc[val_ids]
try:
    calibration_ids, holdout_ids = train_test_split(
        val_ids,
        test_size=0.40,
        random_state=2026,
        stratify=val_combo.loc[val_ids]
    )
    INTERNAL_SPLIT_NOTE = "stratified by four-class presence combination"
except ValueError as exc:
    warnings.warn(f"Could not stratify internal validation split: {exc}. Using deterministic fallback.")
    calibration_ids, holdout_ids = train_test_split(
        val_ids,
        test_size=0.40,
        random_state=2026
    )
    INTERNAL_SPLIT_NOTE = "deterministic random fallback"

calibration_ids = np.asarray(calibration_ids, dtype=str)
holdout_ids = np.asarray(holdout_ids, dtype=str)
if set(calibration_ids).intersection(set(holdout_ids)):
    raise RuntimeError("Calibration and holdout IDs overlap.")
if set(calibration_ids).union(set(holdout_ids)) != set(val_ids):
    raise RuntimeError("Calibration/holdout split does not cover the full validation set.")

# CAL is split once more: CAL_TUNE selects numeric parameters; CAL_AUDIT only
# accepts/rejects the final high-level soft-gate/conflict policy. HOLDOUT stays untouched.
try:
    cal_tune_ids, cal_audit_ids = train_test_split(
        calibration_ids,
        test_size=0.25,
        random_state=2027,
        stratify=label_combo.loc[calibration_ids]
    )
    CAL_AUDIT_SPLIT_NOTE = "stratified CAL_TUNE/CAL_AUDIT split"
except ValueError as exc:
    warnings.warn(f"Could not stratify CAL_TUNE/CAL_AUDIT: {exc}. Using deterministic fallback.")
    cal_tune_ids, cal_audit_ids = train_test_split(
        calibration_ids, test_size=0.25, random_state=2027
    )
    CAL_AUDIT_SPLIT_NOTE = "deterministic CAL_TUNE/CAL_AUDIT fallback"

val_index_by_id = {image_id: index for index, image_id in enumerate(val_ids)}
CAL_IDX = np.asarray([val_index_by_id[x] for x in cal_tune_ids], dtype=np.int64)
CAL_AUDIT_IDX = np.asarray([val_index_by_id[x] for x in cal_audit_ids], dtype=np.int64)
HOLDOUT_IDX = np.asarray([val_index_by_id[x] for x in holdout_ids], dtype=np.int64)
ALL_VAL_IDX = np.arange(len(val_ids), dtype=np.int64)

if set(CAL_IDX).intersection(set(CAL_AUDIT_IDX)) or set(CAL_IDX).intersection(set(HOLDOUT_IDX)) or set(CAL_AUDIT_IDX).intersection(set(HOLDOUT_IDX)):
    raise RuntimeError("CAL_TUNE, CAL_AUDIT and HOLDOUT overlap.")
if set(CAL_IDX).union(set(CAL_AUDIT_IDX), set(HOLDOUT_IDX)) != set(ALL_VAL_IDX):
    raise RuntimeError("CAL_TUNE/CAL_AUDIT/HOLDOUT do not cover the validation split exactly.")

val_positive_by_class = presence.loc[val_ids, CLASSES].sum(axis=0).astype(int).to_dict()
audit_positive_by_class = presence.loc[cal_audit_ids, CLASSES].sum(axis=0).astype(int).to_dict()
print("Competition data root:", DATA_ROOT)
print("Unique images:", len(all_image_ids))
print("Train / validation:", len(train_ids), "/", len(val_ids))
print("CAL_TUNE / CAL_AUDIT / HOLDOUT:", len(CAL_IDX), "/", len(CAL_AUDIT_IDX), "/", len(HOLDOUT_IDX))
print("CAL audit split:", CAL_AUDIT_SPLIT_NOTE)
print("CAL_AUDIT positive pairs by class:", audit_positive_by_class)
print("Internal split:", INTERNAL_SPLIT_NOTE)
print("Validation positive pairs by class:", val_positive_by_class)
print("Validation image-class pairs:", len(val_ids) * N_CLASSES)

In [ ]:
def rle_decode(rle, height=HEIGHT, width=WIDTH):
    flat = np.zeros(height * width, dtype=np.uint8)
    if rle is None or str(rle).strip() == "":
        return flat.reshape((height, width), order="F")
    values = np.asarray(str(rle).split(), dtype=np.int64)
    if len(values) % 2:
        raise ValueError("RLE must contain start-length pairs.")
    starts = values[0::2] - 1
    lengths = values[1::2]
    for start, length in zip(starts, lengths):
        if start < 0 or length <= 0 or start + length > height * width:
            raise ValueError("RLE run is outside the 128x800 image.")
        flat[start:start + length] = 1
    return flat.reshape((height, width), order="F")

mask_lookup = {
    (row.image_id, row.defect_type): row.mask_rle
    for row in train_df[["image_id", "defect_type", "mask_rle"]].itertuples(index=False)
}

# Match the original training preprocessing for every saved model family.
# V4, V5, V6 and MiT-B4 training datasets all used bilinear resizing.
class SurfaceGuardValidationDataset(Dataset):
    def __init__(self, image_ids, data_root, resample):
        self.image_ids = list(map(str, image_ids))
        self.data_root = Path(data_root)
        self.resample = resample

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        image_path = self.data_root / "train" / f"{image_id}.jpg"
        with Image.open(image_path) as source:
            image = source.convert("L").resize((WIDTH, HEIGHT), resample=self.resample)
            image = np.asarray(image, dtype=np.float32) / 255.0

        mean = float(image.mean())
        std = max(float(image.std()), 0.05)
        image = np.ascontiguousarray((image - mean) / std, dtype=np.float32)

        masks = np.stack([
            rle_decode(mask_lookup.get((image_id, class_name), ""))
            for class_name in CLASSES
        ], axis=0).astype(np.uint8)

        return torch.from_numpy(image).unsqueeze(0), torch.from_numpy(masks), image_id


MAIN_RESAMPLE = Image.Resampling.BILINEAR
MIT_RESAMPLE = Image.Resampling.BILINEAR

val_dataset_main = SurfaceGuardValidationDataset(
    val_ids, DATA_ROOT, MAIN_RESAMPLE
)
val_dataset_mit = SurfaceGuardValidationDataset(
    val_ids, DATA_ROOT, MIT_RESAMPLE
)

val_loader_main = DataLoader(
    val_dataset_main,
    batch_size=MAIN_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False
)
val_loader_mit = DataLoader(
    val_dataset_mit,
    batch_size=MIT_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False
)

print("Main validation images:", len(val_dataset_main))
print("MiT validation images:", len(val_dataset_mit))
print("Main batches:", len(val_loader_main))
print("MiT batches:", len(val_loader_mit))

In [ ]:
# Inference-only checkpoint loading. No optimizer, scheduler, loss or training loop is created.
# Checkpoints are loaded one at a time; optimizer/scaler states are discarded immediately.

def torch_load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:  # compatibility with older Kaggle PyTorch versions
        return torch.load(path, map_location="cpu")


def safe_model_to_device(model, label):
    """Place a model on the selected device, falling back on CUDA OOM."""
    try:
        model = model.to(DEVICE)
    except RuntimeError as exc:
        if DEVICE.type == "cuda" and _is_cuda_oom_error(exc):
            switch_to_cpu_fallback(exc, extra_models=(model,))
            model = model.to(torch.device("cpu"))
        else:
            raise
    model.eval()
    return model


def verify_checkpoint_metadata(checkpoint, label):
    saved_classes = checkpoint.get("classes")
    if saved_classes is not None and list(saved_classes) != CLASSES:
        raise ValueError(
            f"{label} checkpoint class order mismatch: {list(saved_classes)} != {CLASSES}"
        )
    saved_h, saved_w = checkpoint.get("height"), checkpoint.get("width")
    if saved_h is not None and int(saved_h) != HEIGHT:
        raise ValueError(f"{label} checkpoint height mismatch: {saved_h}")
    if saved_w is not None and int(saved_w) != WIDTH:
        raise ValueError(f"{label} checkpoint width mismatch: {saved_w}")
    if "model_state_dict" not in checkpoint:
        raise KeyError(f"{label} checkpoint has no model_state_dict.")


def load_smp_checkpoint(path, model, label):
    checkpoint = torch_load_checkpoint(path)
    verify_checkpoint_metadata(checkpoint, label)
    metadata = {
        "epoch": checkpoint.get("epoch"),
        "best_val_loss": checkpoint.get("best_val_loss"),
        "mean_dice": checkpoint.get("mean_dice")
    }
    result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    if result.missing_keys or result.unexpected_keys:
        raise RuntimeError(f"{label} strict checkpoint load was not clean: {result}")
    del checkpoint
    gc.collect()
    model = safe_model_to_device(model, label)
    
    print(f"{label} loaded | epoch={metadata['epoch']} | parameters={sum(p.numel() for p in model.parameters()):,}")
    return model, metadata


# V4 — U-Net with ResNet34 encoder.
model_v4 = smp.Unet(
    encoder_name="resnet34", encoder_weights=None,
    in_channels=1, classes=N_CLASSES, activation=None
)
model_v4, META_V4 = load_smp_checkpoint(V4_PATH, model_v4, "V4 U-Net/ResNet34")

# V5 — SegFormer-B0; fetch config only, then strictly load the attached four-class weights.
V5_MODEL_ID = "nvidia/segformer-b0-finetuned-ade-512-512"
config_v5 = SegformerConfig.from_pretrained(V5_MODEL_ID)
config_v5.num_labels = N_CLASSES
config_v5.id2label = {i: name for i, name in enumerate(CLASSES)}
config_v5.label2id = {name: i for i, name in enumerate(CLASSES)}
model_v5 = SegformerForSemanticSegmentation(config_v5)
checkpoint_v5 = torch_load_checkpoint(V5_PATH)
verify_checkpoint_metadata(checkpoint_v5, "V5 SegFormer-B0")
META_V5 = {"epoch": checkpoint_v5.get("epoch"), "mean_dice": checkpoint_v5.get("mean_dice")}
result_v5 = model_v5.load_state_dict(checkpoint_v5["model_state_dict"], strict=True)
if result_v5.missing_keys or result_v5.unexpected_keys:
    raise RuntimeError(f"V5 strict checkpoint load was not clean: {result_v5}")
del checkpoint_v5
model_v5 = safe_model_to_device(model_v5, "V5 SegFormer-B0")
gc.collect()

print(f"V5 SegFormer-B0 loaded | epoch={META_V5['epoch']} | parameters={sum(p.numel() for p in model_v5.parameters()):,}")

# V6 — DeepLabV3+ with ResNet50 encoder.
model_v6 = smp.DeepLabV3Plus(
    encoder_name="resnet50", encoder_weights=None,
    in_channels=1, classes=N_CLASSES, activation=None
)
model_v6, META_V6 = load_smp_checkpoint(
    V6_PATH, model_v6, "V6 DeepLabV3+/ResNet50"
)

# MiT-B4 V2 checkpoint and report determine the model ID and saved inference mode.
with open(MIT_REPORT_PATH, "r") as f:
    mit_report = json.load(f)
MIT_MODEL_ID = mit_report.get(
    "model", "nvidia/segformer-b4-finetuned-ade-512-512"
)
MIT_INFERENCE_MODE = mit_report.get("selected_inference_mode")
if MIT_INFERENCE_MODE not in {"four_way_flip_tta", "no_tta"}:
    raise ValueError(
        "MiT-B4 report must have selected_inference_mode equal to "
        "'four_way_flip_tta' or 'no_tta'."
    )
MIT_USE_TTA = MIT_INFERENCE_MODE == "four_way_flip_tta"

checkpoint_mit = torch_load_checkpoint(MIT_PATH)
verify_checkpoint_metadata(checkpoint_mit, "MiT-B4")
checkpoint_model_id = checkpoint_mit.get("model_id")
if checkpoint_model_id and checkpoint_model_id != MIT_MODEL_ID:
    raise ValueError(
        f"MiT model ID mismatch: checkpoint={checkpoint_model_id}, report={MIT_MODEL_ID}"
    )
META_MIT = {"epoch": checkpoint_mit.get("epoch"), "model_id": MIT_MODEL_ID}
config_mit = SegformerConfig.from_pretrained(MIT_MODEL_ID)
config_mit.num_labels = N_CLASSES
config_mit.id2label = {i: name for i, name in enumerate(CLASSES)}
config_mit.label2id = {name: i for i, name in enumerate(CLASSES)}
model_mit = SegformerForSemanticSegmentation(config_mit)
result_mit = model_mit.load_state_dict(
    checkpoint_mit["model_state_dict"], strict=True
)
if result_mit.missing_keys or result_mit.unexpected_keys:
    raise RuntimeError(f"MiT-B4 strict checkpoint load was not clean: {result_mit}")
del checkpoint_mit
model_mit = safe_model_to_device(model_mit, "MiT-B4")
gc.collect()



# Optional Colab V7 auxiliary-classifier models. Their four checkpoints were
# created by the executed Colab V7 notebook and saved as model_state_dict bundles.
COLAB_V7_GROUP_WEIGHTS = {
    "model_colab_a": 0.8,
    "model_colab_b": 1.0,
    "model_colab_c": 1.2,
    "model_colab_d": 1.0,
}
COLAB_V7_MODEL_SPECS = []

if COLAB_V7_AVAILABLE:
    def build_colab_aux_unet(encoder_name, num_classes):
        return smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=None,  # All trained weights are loaded strictly below.
            in_channels=1,
            classes=num_classes,
            activation=None,
            aux_params=dict(pooling="avg", dropout=0.3, activation=None, classes=num_classes),
        )

    def build_colab_aux_deeplab(num_classes):
        return smp.DeepLabV3Plus(
            encoder_name="resnet50",
            encoder_weights=None,
            in_channels=1,
            classes=num_classes,
            activation=None,
            aux_params=dict(pooling="avg", dropout=0.3, activation=None, classes=num_classes),
        )

    class ColabV7SegFormerWithClassifier(torch.nn.Module):
        def __init__(self, model_name, num_classes):
            super().__init__()
            config = SegformerConfig.from_pretrained(model_name)
            config.num_labels = num_classes
            config.id2label = {i: name for i, name in enumerate(CLASSES)}
            config.label2id = {name: i for i, name in enumerate(CLASSES)}
            self.base = SegformerForSemanticSegmentation(config)
            self.classifier_head = torch.nn.Sequential(
                torch.nn.Linear(num_classes, 64),
                torch.nn.ReLU(inplace=True),
                torch.nn.Linear(64, num_classes),
            )

        def forward(self, pixel_values, target_size):
            seg_logits = self.base(pixel_values=pixel_values).logits
            seg_logits = F.interpolate(
                seg_logits.float(), size=target_size, mode="bilinear", align_corners=False
            )
            pooled = torch.amax(seg_logits, dim=(-2, -1))
            cls_logits = self.classifier_head(pooled)
            return seg_logits, cls_logits

    def colab_v7_unet_forward(model, images):
        seg_logits, cls_logits = model(images)
        return seg_logits, cls_logits

    def colab_v7_segformer_forward(model, images):
        rgb_images = images.repeat(1, 3, 1, 1)
        return model(rgb_images, target_size=(HEIGHT, WIDTH))

    def load_colab_v7_checkpoint(path, model, label):
        checkpoint = torch_load_checkpoint(path)
        verify_checkpoint_metadata(checkpoint, label)
        result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        if result.missing_keys or result.unexpected_keys:
            raise RuntimeError(f"{label} strict checkpoint load failed: {result}")
        metadata = {
            "weighted_score": checkpoint.get("weighted_score"),
            "run_name": checkpoint.get("run_name"),
        }
        del checkpoint
        gc.collect()
        model = safe_model_to_device(model, label)
        print(f"{label} loaded | metadata={metadata} | parameters={sum(p.numel() for p in model.parameters()):,}")
        return model

    # Build without downloading ImageNet tensors: strict checkpoint loading replaces
    # every trained parameter and preserves the Colab V7 architecture.
    model_colab_a = load_colab_v7_checkpoint(
        COLAB_V7_PATHS["a_scratch"],
        build_colab_aux_unet("resnet34", N_CLASSES),
        "Colab V7 Model A — scratch ResNet34 U-Net + classifier",
    )
    model_colab_b = load_colab_v7_checkpoint(
        COLAB_V7_PATHS["b_pretrained"],
        build_colab_aux_unet("resnet34", N_CLASSES),
        "Colab V7 Model B — pretrained ResNet34 U-Net + classifier",
    )
    model_colab_c = load_colab_v7_checkpoint(
        COLAB_V7_PATHS["c_segformer_b0"],
        ColabV7SegFormerWithClassifier(
            "nvidia/segformer-b0-finetuned-ade-512-512", N_CLASSES
        ),
        "Colab V7 Model C — SegFormer MiT-B0 + classifier",
    )
    model_colab_d = load_colab_v7_checkpoint(
        COLAB_V7_PATHS["d_deeplabv3plus"],
        build_colab_aux_deeplab(N_CLASSES),
        "Colab V7 Model D — DeepLabV3+/ResNet50 + classifier",
    )

    COLAB_V7_MODEL_SPECS = [
        (colab_v7_unet_forward, model_colab_a, COLAB_V7_GROUP_WEIGHTS["model_colab_a"]),
        (colab_v7_unet_forward, model_colab_b, COLAB_V7_GROUP_WEIGHTS["model_colab_b"]),
        (colab_v7_segformer_forward, model_colab_c, COLAB_V7_GROUP_WEIGHTS["model_colab_c"]),
        (colab_v7_unet_forward, model_colab_d, COLAB_V7_GROUP_WEIGHTS["model_colab_d"]),
    ]
    print("\nCOLAB V7 AUXILIARY CHECKPOINTS LOADED — INFERENCE ONLY")
else:
    print("\nColab V7 auxiliary models skipped; V13 four-model fallback remains active.")

models = {
    "V4_U-Net_ResNet34": model_v4,
    "V5_SegFormer_B0": model_v5,
    "V6_DeepLabV3Plus_ResNet50": model_v6,
    "MiT_B4_SegFormer": model_mit
}
if COLAB_V7_AVAILABLE:
    models.update({
        "ColabV7_A_Scratch_UNet": model_colab_a,
        "ColabV7_B_Pretrained_UNet": model_colab_b,
        "ColabV7_C_SegFormer_B0": model_colab_c,
        "ColabV7_D_DeepLabV3Plus": model_colab_d,
    })

print("\nFOUR CHECKPOINTS LOADED SUCCESSFULLY — INFERENCE ONLY")
print("V4 metadata:", META_V4)
print("V5 metadata:", META_V5)
print("V6 metadata:", META_V6)
print("MiT-B4 metadata:", META_MIT)
print("MiT-B4 inference mode recorded in saved V2 report:", MIT_INFERENCE_MODE)
print("The effective runtime mode is resolved in the next cell according to the selected device.")
print("All models on:", DEVICE)


In [ ]:
# GPU mode uses the full four-way flip TTA experiment. CPU-only mode uses one
# unflipped pass for all models to keep a fallback run practical. Validation and
# test inference always use the same selected mode within a run.
if DEVICE.type == "cuda":
    MAIN_INFERENCE_MODE = "four_way_flip_tta"
    MAIN_TTA_TRANSFORMS = [(False, False), (True, False), (False, True), (True, True)]
else:
    MAIN_INFERENCE_MODE = "no_tta_cpu_fallback"
    MAIN_TTA_TRANSFORMS = [(False, False)]
    MIT_USE_TTA = False
    MIT_INFERENCE_MODE = "no_tta_cpu_fallback"


def autocast_context():
    if DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def resize_segformer_logits(logits):
    if logits.shape[-2:] != (HEIGHT, WIDTH):
        logits = F.interpolate(
            logits.float(), size=(HEIGHT, WIDTH), mode="bilinear", align_corners=False
        )
    return logits.float()


def _predict_main_models_impl(images):
    images = images.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
    sums = [None, None, None]
    for hflip, vflip in MAIN_TTA_TRANSFORMS:
        transformed = images
        if hflip:
            transformed = torch.flip(transformed, dims=[3])
        if vflip:
            transformed = torch.flip(transformed, dims=[2])
        with autocast_context():
            logits_v4 = model_v4(transformed)
            probs_v4 = torch.sigmoid(logits_v4.float())
            del logits_v4
            rgb_images = transformed.repeat(1, 3, 1, 1)
            logits_v5 = model_v5(pixel_values=rgb_images).logits
            logits_v5 = resize_segformer_logits(logits_v5)
            probs_v5 = torch.sigmoid(logits_v5.float())
            del logits_v5, rgb_images
            logits_v6 = model_v6(transformed)
            probs_v6 = torch.sigmoid(logits_v6.float())
            del logits_v6
        current = [probs_v4, probs_v5, probs_v6]
        if vflip:
            current = [torch.flip(p, dims=[2]) for p in current]
        if hflip:
            current = [torch.flip(p, dims=[3]) for p in current]
        for model_index, probabilities in enumerate(current):
            sums[model_index] = probabilities if sums[model_index] is None else sums[model_index] + probabilities
        del transformed, probs_v4, probs_v5, probs_v6, current
    return tuple(total / len(MAIN_TTA_TRANSFORMS) for total in sums)


@torch.inference_mode()
def predict_main_models(images):
    try:
        return _predict_main_models_impl(images)
    except RuntimeError as exc:
        if DEVICE.type == "cuda" and _is_cuda_oom_error(exc):
            switch_to_cpu_fallback(exc)
            return _predict_main_models_impl(images)
        raise


def _predict_mit_model_impl(images):
    images = images.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
    def one_pass(x):
        rgb = x.repeat(1, 3, 1, 1)
        with autocast_context():
            logits = model_mit(pixel_values=rgb).logits
            logits = resize_segformer_logits(logits)
            return torch.sigmoid(logits.float())
    if not MIT_USE_TTA:
        return one_pass(images)
    total = None
    transforms = [(False, False), (True, False), (False, True), (True, True)]
    for hflip, vflip in transforms:
        transformed = images
        if hflip:
            transformed = torch.flip(transformed, dims=[3])
        if vflip:
            transformed = torch.flip(transformed, dims=[2])
        probs = one_pass(transformed)
        if vflip:
            probs = torch.flip(probs, dims=[2])
        if hflip:
            probs = torch.flip(probs, dims=[3])
        total = probs if total is None else total + probs
    return total / len(transforms)


@torch.inference_mode()
def predict_mit_model(images):
    try:
        return _predict_mit_model_impl(images)
    except RuntimeError as exc:
        if DEVICE.type == "cuda" and _is_cuda_oom_error(exc):
            switch_to_cpu_fallback(exc)
            return _predict_mit_model_impl(images)
        raise




@torch.inference_mode()
def _predict_colab_v7_group_impl(images):
    """Return weighted mean segmentation and image-level class probabilities."""
    if not COLAB_V7_AVAILABLE:
        return None, None
    images = images.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
    seg_total = None
    cls_total = None
    weight_total = 0.0
    transforms = MAIN_TTA_TRANSFORMS

    for forward_fn, model, model_weight in COLAB_V7_MODEL_SPECS:
        model_seg_total = None
        model_cls_total = None
        for hflip, vflip in transforms:
            transformed = images
            if hflip:
                transformed = torch.flip(transformed, dims=[3])
            if vflip:
                transformed = torch.flip(transformed, dims=[2])
            with autocast_context():
                seg_logits, cls_logits = forward_fn(model, transformed)
                if seg_logits.shape[-2:] != (HEIGHT, WIDTH):
                    seg_logits = F.interpolate(
                        seg_logits.float(), size=(HEIGHT, WIDTH), mode="bilinear", align_corners=False
                    )
                seg_probs = torch.sigmoid(seg_logits.float())
                cls_probs = torch.sigmoid(cls_logits.float())
            if vflip:
                seg_probs = torch.flip(seg_probs, dims=[2])
            if hflip:
                seg_probs = torch.flip(seg_probs, dims=[3])
            model_seg_total = seg_probs if model_seg_total is None else model_seg_total + seg_probs
            model_cls_total = cls_probs if model_cls_total is None else model_cls_total + cls_probs
            del transformed, seg_logits, cls_logits, seg_probs, cls_probs
        model_seg = model_seg_total / len(transforms)
        model_cls = model_cls_total / len(transforms)
        seg_total = model_weight * model_seg if seg_total is None else seg_total + model_weight * model_seg
        cls_total = model_weight * model_cls if cls_total is None else cls_total + model_weight * model_cls
        weight_total += model_weight
        del model_seg_total, model_cls_total, model_seg, model_cls
    return seg_total / weight_total, cls_total / weight_total


@torch.inference_mode()
def predict_colab_v7_group(images):
    if not COLAB_V7_AVAILABLE:
        return None, None
    try:
        seg, cls = _predict_colab_v7_group_impl(images)
    except RuntimeError as exc:
        if DEVICE.type == "cuda" and _is_cuda_oom_error(exc):
            switch_to_cpu_fallback(
                exc,
                extra_models=[spec[1] for spec in COLAB_V7_MODEL_SPECS],
            )
            seg, cls = _predict_colab_v7_group_impl(images)
        else:
            raise
    if not torch.isfinite(seg).all() or not torch.isfinite(cls).all():
        raise RuntimeError("Colab V7 auxiliary prediction contains non-finite values.")
    if seg.min().item() < 0 or seg.max().item() > 1 or cls.min().item() < 0 or cls.max().item() > 1:
        raise RuntimeError("Colab V7 probabilities are outside [0, 1].")
    return seg, cls

# One-batch inference check before generating any large prediction caches.
smoke_images, smoke_masks, smoke_ids = next(iter(val_loader_main))
smoke_v4, smoke_v5, smoke_v6 = predict_main_models(smoke_images)
smoke_mit_images, _, _ = next(iter(val_loader_mit))
smoke_mit = predict_mit_model(smoke_mit_images)

for name, tensor in [
    ("V4", smoke_v4), ("V5", smoke_v5), ("V6", smoke_v6), ("MiT-B4", smoke_mit)
]:
    expected = (len(tensor), N_CLASSES, HEIGHT, WIDTH)
    if tuple(tensor.shape) != expected:
        raise RuntimeError(f"{name} output shape is invalid: {tuple(tensor.shape)} != {expected}")
    if not torch.isfinite(tensor).all():
        raise RuntimeError(f"{name} output contains non-finite values.")
    if tensor.min().item() < 0 or tensor.max().item() > 1:
        raise RuntimeError(f"{name} probabilities are outside [0, 1].")
    print(name, tuple(tensor.shape), "range:", float(tensor.min()), float(tensor.max()))

print("Inference smoke test: PASSED")

if COLAB_V7_AVAILABLE:
    smoke_colab_seg, smoke_colab_cls = predict_colab_v7_group(smoke_images)
    if tuple(smoke_colab_seg.shape) != (len(smoke_images), N_CLASSES, HEIGHT, WIDTH):
        raise RuntimeError(f"Colab V7 segmentation shape is invalid: {tuple(smoke_colab_seg.shape)}")
    if tuple(smoke_colab_cls.shape) != (len(smoke_images), N_CLASSES):
        raise RuntimeError(f"Colab V7 classifier shape is invalid: {tuple(smoke_colab_cls.shape)}")
    print("Colab V7 auxiliary group smoke test: PASSED")
    del smoke_colab_seg, smoke_colab_cls
print("Execution device:", DEVICE, "|", GPU_NAME)
print("Main inference mode:", MAIN_INFERENCE_MODE, "| TTA passes:", len(MAIN_TTA_TRANSFORMS))
print("MiT inference mode:", MIT_INFERENCE_MODE)
del smoke_images, smoke_masks, smoke_ids, smoke_v4, smoke_v5, smoke_v6, smoke_mit_images, smoke_mit
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
N_VAL = len(val_ids)
VAL_SHAPE = (N_VAL, N_CLASSES, HEIGHT, WIDTH)

val_v4_path = CACHE_DIR / "val_v4.npy"
val_v5_path = CACHE_DIR / "val_v5.npy"
val_v6_path = CACHE_DIR / "val_v6.npy"
val_mit_path = CACHE_DIR / "val_mit_b4.npy"
val_masks_path = CACHE_DIR / "val_masks.npy"
val_colab_seg_path = CACHE_DIR / "val_colab_v7_seg.npy"
val_colab_cls_path = CACHE_DIR / "val_colab_v7_cls.npy"

# Four baseline segmentation maps + masks; optional Colab branch adds one ensemble map + classifier scores.
cache_factor_bytes = (2 * 4 + 1) + (2 + 2 / (HEIGHT * WIDTH) if COLAB_V7_AVAILABLE else 0)
cache_required_bytes = int(np.prod(VAL_SHAPE) * cache_factor_bytes)
cache_free_bytes = shutil.disk_usage(CACHE_DIR).free
print("Estimated validation cache:", round(cache_required_bytes / (1024 ** 3), 2), "GiB")
print("Free space at cache location:", round(cache_free_bytes / (1024 ** 3), 2), "GiB")
if cache_free_bytes < int(cache_required_bytes * 1.15):
    raise RuntimeError(
        "Insufficient free space for validation probability caches. "
        "Clear temporary Kaggle files or use a runtime with more disk space before proceeding."
    )

# Fresh V19-specific cache files prevent stale V13/V14 predictions being reused.
for stale in [val_v4_path, val_v5_path, val_v6_path, val_mit_path, val_masks_path,
              val_colab_seg_path, val_colab_cls_path]:
    if stale.exists():
        stale.unlink()

val_v4_probs = np.lib.format.open_memmap(val_v4_path, mode="w+", dtype=np.float16, shape=VAL_SHAPE)
val_v5_probs = np.lib.format.open_memmap(val_v5_path, mode="w+", dtype=np.float16, shape=VAL_SHAPE)
val_v6_probs = np.lib.format.open_memmap(val_v6_path, mode="w+", dtype=np.float16, shape=VAL_SHAPE)
val_mit_probs = np.lib.format.open_memmap(val_mit_path, mode="w+", dtype=np.float16, shape=VAL_SHAPE)
val_masks = np.lib.format.open_memmap(val_masks_path, mode="w+", dtype=np.uint8, shape=VAL_SHAPE)
if COLAB_V7_AVAILABLE:
    val_colab_seg_probs = np.lib.format.open_memmap(
        val_colab_seg_path, mode="w+", dtype=np.float16, shape=VAL_SHAPE
    )
    val_colab_cls_probs = np.lib.format.open_memmap(
        val_colab_cls_path, mode="w+", dtype=np.float16, shape=(N_VAL, N_CLASSES)
    )

print("Generating validation predictions for the proven Kaggle group...")
if COLAB_V7_AVAILABLE:
    print("Also generating the four-model Colab V7 auxiliary segmentation + classifier ensemble...")
cursor = 0
with torch.inference_mode():
    for images, masks, image_ids in tqdm(val_loader_main, desc="Validation: Kaggle + optional Colab V7"):
        p4_batch, p5_batch, p6_batch = predict_main_models(images)
        batch_size = len(image_ids)
        end = cursor + batch_size
        val_v4_probs[cursor:end] = p4_batch.detach().cpu().numpy().astype(np.float16)
        val_v5_probs[cursor:end] = p5_batch.detach().cpu().numpy().astype(np.float16)
        val_v6_probs[cursor:end] = p6_batch.detach().cpu().numpy().astype(np.float16)
        val_masks[cursor:end] = masks.numpy().astype(np.uint8)
        if COLAB_V7_AVAILABLE:
            colab_seg_batch, colab_cls_batch = predict_colab_v7_group(images)
            val_colab_seg_probs[cursor:end] = colab_seg_batch.detach().cpu().numpy().astype(np.float16)
            val_colab_cls_probs[cursor:end] = colab_cls_batch.detach().cpu().numpy().astype(np.float16)
            del colab_seg_batch, colab_cls_batch
        cursor = end
        del p4_batch, p5_batch, p6_batch

if cursor != N_VAL:
    raise RuntimeError(f"Main validation prediction count mismatch: {cursor} != {N_VAL}")

for array in [val_v4_probs, val_v5_probs, val_v6_probs, val_masks]:
    array.flush()
if COLAB_V7_AVAILABLE:
    val_colab_seg_probs.flush()
    val_colab_cls_probs.flush()

print("Kaggle V4/V5/V6 validation predictions saved:", cursor)
if COLAB_V7_AVAILABLE:
    print("Colab V7 ensemble segmentation and classifier predictions saved:", cursor)
print("Validation arrays:", VAL_SHAPE)
print("Cache directory:", CACHE_DIR)


In [ ]:
print("MiT-B4 validation inference mode:", MIT_INFERENCE_MODE)
cursor = 0
with torch.inference_mode():
    for images, masks, image_ids in tqdm(val_loader_mit, desc="Validation: MiT-B4"):
        mit_batch = predict_mit_model(images)
        batch_size = len(image_ids)
        end = cursor + batch_size
        val_mit_probs[cursor:end] = mit_batch.detach().cpu().numpy().astype(np.float16)
        cursor = end

if cursor != N_VAL:
    raise RuntimeError(f"MiT validation prediction count mismatch: {cursor} != {N_VAL}")

val_mit_probs.flush()

# Reopen disk-backed arrays read-only. They remain memory mapped, not copied into RAM.
val_v4_probs = np.load(val_v4_path, mmap_mode="r")
val_v5_probs = np.load(val_v5_path, mmap_mode="r")
val_v6_probs = np.load(val_v6_path, mmap_mode="r")
val_mit_probs = np.load(val_mit_path, mmap_mode="r")
val_masks = np.load(val_masks_path, mmap_mode="r")

for name, array in [
    ("V4", val_v4_probs), ("V5", val_v5_probs), ("V6", val_v6_probs),
    ("MiT-B4", val_mit_probs), ("Ground truth", val_masks)
]:
    if array.shape != VAL_SHAPE:
        raise RuntimeError(f"{name} validation array has wrong shape: {array.shape}")
    if name != "Ground truth":
        # Check finite values in small chunks to avoid reading/copying all arrays at once.
        for start in range(0, N_VAL, 16):
            if not np.isfinite(array[start:start + 16]).all():
                raise RuntimeError(f"{name} contains non-finite probabilities.")
    print(name, array.shape, array.dtype)

print("All four validation caches are ready.")

In [ ]:
# Threshold grid used for calibration. Histogram bins let us evaluate a finer 0.005 threshold grid
# without repeatedly materialising full-resolution binary masks.
THRESHOLD_GRID = np.round(np.arange(0.01, 0.951, 0.005), 3).astype(np.float32)
N_THRESHOLD_BINS = len(THRESHOLD_GRID) + 1
CAL_CHUNK_SIZE = 8

# A blend is always [V4, V5, V6, MiT-B4] and must sum to one.
def validate_blend_weights(weights):
    weights = tuple(float(x) for x in weights)
    if len(weights) != 4 or any((not np.isfinite(x)) or x < -1e-9 for x in weights):
        raise ValueError(f"Invalid four-model blend weights: {weights}")
    if not np.isclose(sum(weights), 1.0, atol=1e-6):
        raise ValueError(f"Blend weights must sum to one: {weights}")
    return weights


def validate_blend_weight_matrix(matrix):
    """Validate a [class, model] matrix; each class row must sum to one."""
    matrix = np.asarray(matrix, dtype=np.float64)
    expected_shape = (N_CLASSES, 4)
    if matrix.shape != expected_shape:
        raise ValueError(f"Class-weight matrix must have shape {expected_shape}; got {matrix.shape}")
    if not np.isfinite(matrix).all() or np.any(matrix < -1e-9):
        raise ValueError("Class-weight matrix contains invalid or negative weights.")
    row_sums = matrix.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-6):
        raise ValueError(f"Each class weight row must sum to one; got {row_sums}")
    return matrix


def combine_cached_probabilities(row_indices, class_idx, weights):
    # Accept either one global [V4,V5,V6,MiT] vector or one [class,model] matrix.
    weight_array = np.asarray(weights, dtype=np.float64)
    if weight_array.ndim == 2:
        weight_array = validate_blend_weight_matrix(weight_array)[class_idx]
    else:
        weight_array = np.asarray(validate_blend_weights(weight_array), dtype=np.float64)
    w4, w5, w6, wm = weight_array
    p4 = np.asarray(val_v4_probs[row_indices, class_idx], dtype=np.float32)
    p5 = np.asarray(val_v5_probs[row_indices, class_idx], dtype=np.float32)
    p6 = np.asarray(val_v6_probs[row_indices, class_idx], dtype=np.float32)
    pm = np.asarray(val_mit_probs[row_indices, class_idx], dtype=np.float32)
    return w4 * p4 + w5 * p5 + w6 * p6 + wm * pm


# Histogram boundary self-test: side="right" plus suffix index j+1 implements p >= t.
for _j in (0, 40, len(THRESHOLD_GRID) - 1):
    _t = THRESHOLD_GRID[_j]
    _toy = np.array([
        np.nextafter(_t, np.float32(0.0)), _t,
        np.nextafter(_t, np.float32(1.0))
    ], dtype=np.float32)
    _bins = np.searchsorted(THRESHOLD_GRID, _toy, side="right")
    _predicted = _bins >= (_j + 1)
    assert _predicted.tolist() == [False, True, True], (float(_t), _toy, _bins, _predicted)
del _j, _t, _toy, _bins, _predicted
print("Threshold histogram >= boundary self-test: PASSED")

# Count ground-truth pixels per image/class once.
gt_counts = np.zeros((N_VAL, N_CLASSES), dtype=np.uint32)
for start in range(0, N_VAL, 16):
    end = min(start + 16, N_VAL)
    block = np.asarray(val_masks[start:end])
    gt_counts[start:end] = np.count_nonzero(block, axis=(2, 3)).astype(np.uint32)
del block
gc.collect()


def optimise_thresholds_for_weights(weights, indices):
    """Optimise per-class thresholds for global weights or a class-specific weight matrix."""
    weight_array = np.asarray(weights, dtype=np.float64)
    if weight_array.ndim == 2:
        weights = validate_blend_weight_matrix(weight_array)
    else:
        weights = validate_blend_weights(weight_array)
    indices = np.asarray(indices, dtype=np.int64)
    local_gt_counts = gt_counts[indices]
    total_positive = int((local_gt_counts > 0).sum())
    total_empty = int((local_gt_counts == 0).sum())
    if total_positive == 0 or total_empty == 0:
        raise ValueError("Calibration subset must contain positive and empty image-class pairs.")
    total_empty_pixels = total_empty * N_PIXELS

    selected_thresholds = {}
    per_class = {}
    total_score = 0.0

    for class_idx, class_name in enumerate(CLASSES):
        n_rows = len(indices)
        hist_all = np.zeros((n_rows, N_THRESHOLD_BINS), dtype=np.uint32)
        hist_positive = np.zeros_like(hist_all)
        class_gt = local_gt_counts[:, class_idx]
        positive = class_gt > 0
        empty = ~positive
        class_empty_count = int(empty.sum())

        for chunk_start in range(0, n_rows, CAL_CHUNK_SIZE):
            chunk_end = min(chunk_start + CAL_CHUNK_SIZE, n_rows)
            row_indices = indices[chunk_start:chunk_end]
            combined = combine_cached_probabilities(row_indices, class_idx, weights)
            targets = np.asarray(val_masks[row_indices, class_idx] > 0)
            combined_flat = combined.reshape(len(row_indices), -1)
            targets_flat = targets.reshape(len(row_indices), -1)
            bins = np.searchsorted(THRESHOLD_GRID, combined_flat, side="right").astype(np.uint8)

            for local_row in range(len(row_indices)):
                hist_all[chunk_start + local_row] = np.bincount(
                    bins[local_row], minlength=N_THRESHOLD_BINS
                ).astype(np.uint32)
                hist_positive[chunk_start + local_row] = np.bincount(
                    bins[local_row][targets_flat[local_row]], minlength=N_THRESHOLD_BINS
                ).astype(np.uint32)

        suffix_all = np.cumsum(hist_all[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
        suffix_positive = np.cumsum(hist_positive[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
        best_contribution = -1.0
        best_threshold = None
        best_components = None

        for threshold_index, threshold in enumerate(THRESHOLD_GRID):
            predicted_counts = suffix_all[:, threshold_index + 1]
            intersection_counts = suffix_positive[:, threshold_index + 1]

            pair_dice = 2.0 * intersection_counts[positive] / np.maximum(
                predicted_counts[positive] + class_gt[positive], 1
            )
            dice_sum = float(pair_dice.sum(dtype=np.float64))
            class_foreground_dice = dice_sum / int(positive.sum()) if positive.any() else 0.0
            empty_fp_pixels = int(predicted_counts[empty].sum(dtype=np.uint64))
            correct_empty = int((predicted_counts[empty] == 0).sum())
            class_empty_pixels = class_empty_count * N_PIXELS
            contribution = (
                0.70 * dice_sum / total_positive
                + 0.20 * (class_empty_pixels - empty_fp_pixels) / total_empty_pixels
                + 0.10 * correct_empty / total_empty
            )

            if contribution > best_contribution:
                best_contribution = float(contribution)
                best_threshold = float(threshold)
                best_components = {
                    "positive_pairs": int(positive.sum()),
                    "empty_pairs": class_empty_count,
                    "foreground_dice": float(class_foreground_dice),
                    "false_positive_score": float(1.0 - empty_fp_pixels / class_empty_pixels) if class_empty_pixels else float("nan"),
                    "empty_accuracy": float(correct_empty / class_empty_count) if class_empty_count else float("nan"),
                    "global_score_contribution": float(contribution)
                }

        selected_thresholds[class_name] = best_threshold
        per_class[class_name] = {"threshold": best_threshold, **best_components}
        total_score += best_contribution
        del hist_all, hist_positive, suffix_all, suffix_positive
        gc.collect()

    return float(total_score), selected_thresholds, per_class


def evaluate_mix_on_indices(weights, thresholds, indices):
    """Evaluate global or class-specific four-model weights on supplied indices."""
    weight_array = np.asarray(weights, dtype=np.float64)
    if weight_array.ndim == 2:
        weights = validate_blend_weight_matrix(weight_array)
    else:
        weights = validate_blend_weights(weight_array)
    indices = np.asarray(indices, dtype=np.int64)
    positive_dice_sum = 0.0
    positive_pairs = 0
    empty_fp_pixels = 0
    empty_pixels = 0
    correct_empty_pairs = 0
    empty_pairs = 0
    class_metrics = {}

    for class_idx, class_name in enumerate(CLASSES):
        class_dice_sum = 0.0
        class_positive_pairs = 0
        class_empty_fp_pixels = 0
        class_empty_pixels = 0
        class_correct_empty = 0
        class_empty_pairs = 0
        threshold = np.float32(thresholds[class_name])

        for chunk_start in range(0, len(indices), CAL_CHUNK_SIZE):
            row_indices = indices[chunk_start:chunk_start + CAL_CHUNK_SIZE]
            combined = combine_cached_probabilities(row_indices, class_idx, weights)
            prediction = combined >= threshold
            target = np.asarray(val_masks[row_indices, class_idx] > 0)
            target_count = np.count_nonzero(target, axis=(1, 2))
            pred_count = np.count_nonzero(prediction, axis=(1, 2))
            intersection = np.count_nonzero(prediction & target, axis=(1, 2))
            positive = target_count > 0
            empty = ~positive

            if positive.any():
                pair_dice = 2.0 * intersection[positive] / np.maximum(
                    pred_count[positive] + target_count[positive], 1
                )
                class_dice_sum += float(pair_dice.sum())
                class_positive_pairs += int(positive.sum())
            if empty.any():
                class_empty_pairs += int(empty.sum())
                class_empty_fp_pixels += int(pred_count[empty].sum())
                class_empty_pixels += int(empty.sum()) * N_PIXELS
                class_correct_empty += int((pred_count[empty] == 0).sum())

        positive_dice_sum += class_dice_sum
        positive_pairs += class_positive_pairs
        empty_fp_pixels += class_empty_fp_pixels
        empty_pixels += class_empty_pixels
        correct_empty_pairs += class_correct_empty
        empty_pairs += class_empty_pairs
        class_metrics[class_name] = {
            "positive_pairs": class_positive_pairs,
            "empty_pairs": class_empty_pairs,
            "foreground_dice": class_dice_sum / class_positive_pairs if class_positive_pairs else float("nan"),
            "false_positive_score": 1.0 - class_empty_fp_pixels / class_empty_pixels if class_empty_pixels else float("nan"),
            "empty_accuracy": class_correct_empty / class_empty_pairs if class_empty_pairs else float("nan")
        }

    foreground_dice = positive_dice_sum / positive_pairs
    false_positive_score = 1.0 - empty_fp_pixels / empty_pixels
    empty_accuracy = correct_empty_pairs / empty_pairs
    weighted_score = 0.70 * foreground_dice + 0.20 * false_positive_score + 0.10 * empty_accuracy
    return {
        "foreground_dice": float(foreground_dice),
        "false_positive_score": float(false_positive_score),
        "empty_accuracy": float(empty_accuracy),
        "weighted_score": float(weighted_score),
        "positive_pairs": int(positive_pairs),
        "empty_pairs": int(empty_pairs),
        "class_metrics": class_metrics
    }

print("Independent four-model calibration/evaluation functions ready.")


In [ ]:
# Search all four model weights independently on CAL_IDX only.
# Coarse simplex grid: 0.25 increments (35 combinations), plus explicit known anchors.
from itertools import product

coarse_candidates = [
    tuple(round(x * 0.25, 3) for x in parts)
    for parts in product(range(5), repeat=4)
    if sum(parts) == 4
]

if len(coarse_candidates) != 35:
    raise RuntimeError(f"Unexpected four-model coarse grid size: {len(coarse_candidates)}")

V7_WEIGHT_ANCHOR_WEIGHTS = (0.125, 0.125, 0.125, 0.625)
THREE_MODEL_ANCHOR_WEIGHTS = (0.375, 0.375, 0.25, 0.0)

search_rows = []
coarse_best = None
for candidate in tqdm(coarse_candidates, desc="Coarse four-model simplex search"):
    score, thresholds, details = optimise_thresholds_for_weights(candidate, CAL_IDX)
    row = {
        "stage": "coarse",
        "weights": candidate,
        "v4_weight": candidate[0], "v5_weight": candidate[1],
        "v6_weight": candidate[2], "mit_b4_weight": candidate[3],
        "calibration_score": score, "thresholds": thresholds, "per_class": details
    }
    search_rows.append(row)
    if coarse_best is None or score > coarse_best["calibration_score"]:
        coarse_best = row

# Evaluate anchors explicitly. These are not selected using the holdout.
anchor_rows = {}
for label, weights in [
    ("v7_weight_anchor", V7_WEIGHT_ANCHOR_WEIGHTS),
    ("three_model_anchor", THREE_MODEL_ANCHOR_WEIGHTS)
]:
    score, thresholds, details = optimise_thresholds_for_weights(weights, CAL_IDX)
    row = {
        "stage": label, "weights": weights,
        "v4_weight": weights[0], "v5_weight": weights[1],
        "v6_weight": weights[2], "mit_b4_weight": weights[3],
        "calibration_score": score, "thresholds": thresholds, "per_class": details
    }
    search_rows.append(row)
    anchor_rows[label] = row

# Refine around the best coarse/anchor candidate. Deltas sum to zero, preserving the simplex.
search_center = max(search_rows, key=lambda r: r["calibration_score"])
refine_step = 0.125
refine_candidates = set()
for deltas in product([-refine_step, 0.0, refine_step], repeat=4):
    if not np.isclose(sum(deltas), 0.0):
        continue
    candidate = tuple(round(w + d, 3) for w, d in zip(search_center["weights"], deltas))
    if min(candidate) < 0 or not np.isclose(sum(candidate), 1.0):
        continue
    if candidate not in {tuple(r["weights"]) for r in search_rows}:
        refine_candidates.add(candidate)

for candidate in tqdm(sorted(refine_candidates), desc="Local 0.125 four-model refinement"):
    score, thresholds, details = optimise_thresholds_for_weights(candidate, CAL_IDX)
    row = {
        "stage": "refine", "weights": candidate,
        "v4_weight": candidate[0], "v5_weight": candidate[1],
        "v6_weight": candidate[2], "mit_b4_weight": candidate[3],
        "calibration_score": score, "thresholds": thresholds, "per_class": details
    }
    search_rows.append(row)


# V19 fine refinement: transfer 0.05 weight between model pairs around the two
# strongest distinct global candidates. These candidates remain class-aware
# selection candidates, but all final decisions are still made using CAL_IDX.
top_centers = []
seen_centers = set()
for candidate_row in sorted(search_rows, key=lambda r: r["calibration_score"], reverse=True):
    center = tuple(float(x) for x in candidate_row["weights"])
    if center not in seen_centers:
        seen_centers.add(center)
        top_centers.append(center)
    if len(top_centers) == 2:
        break

fine_refine_candidates = set()
FINE_REFINE_STEP = 0.05
for center in top_centers:
    for donor in range(4):
        for receiver in range(4):
            if donor == receiver:
                continue
            candidate = list(center)
            candidate[donor] -= FINE_REFINE_STEP
            candidate[receiver] += FINE_REFINE_STEP
            if min(candidate) < -1e-9:
                continue
            candidate = tuple(round(max(0.0, float(x)), 3) for x in candidate)
            if not np.isclose(sum(candidate), 1.0, atol=1e-6):
                continue
            if candidate not in {tuple(r["weights"]) for r in search_rows}:
                fine_refine_candidates.add(candidate)

fine_refine_rows = []
for candidate in tqdm(
    sorted(fine_refine_candidates),
    desc="V19 fine 0.05 base-model weight refinement"
):
    score, thresholds, details = optimise_thresholds_for_weights(candidate, CAL_IDX)
    row = {
        "stage": "fine_0.05",
        "weights": candidate,
        "v4_weight": candidate[0], "v5_weight": candidate[1],
        "v6_weight": candidate[2], "mit_b4_weight": candidate[3],
        "calibration_score": score, "thresholds": thresholds, "per_class": details
    }
    search_rows.append(row)
    fine_refine_rows.append(row)

fine_blend_search_df = pd.DataFrame([
    {
        "stage": r["stage"], "v4_weight": r["v4_weight"],
        "v5_weight": r["v5_weight"], "v6_weight": r["v6_weight"],
        "mit_b4_weight": r["mit_b4_weight"],
        "calibration_score": r["calibration_score"]
    }
    for r in fine_refine_rows
]).sort_values("calibration_score", ascending=False).reset_index(drop=True)

print("V19 fine candidates evaluated:", len(fine_refine_rows))
if len(fine_blend_search_df):
    print("Top V19 fine-grid candidates:")
    display(fine_blend_search_df.head(10).round(6))
else:
    print("No additional fine-grid candidates were needed; the coarse candidates already covered the local simplex.")

best_calibration_row = max(search_rows, key=lambda r: r["calibration_score"])
SELECTED_WEIGHTS = validate_blend_weights(best_calibration_row["weights"])
SELECTED_V4_WEIGHT, SELECTED_V5_WEIGHT, SELECTED_V6_WEIGHT, SELECTED_MIT_WEIGHT = SELECTED_WEIGHTS
CALIBRATION_THRESHOLDS = dict(best_calibration_row["thresholds"])
V7_WEIGHT_ANCHOR_CALIBRATION_SCORE = float(anchor_rows["v7_weight_anchor"]["calibration_score"])
THREE_MODEL_ANCHOR_CALIBRATION_SCORE = float(anchor_rows["three_model_anchor"]["calibration_score"])

blend_search_df = pd.DataFrame([
    {
        "stage": r["stage"], "v4_weight": r["v4_weight"],
        "v5_weight": r["v5_weight"], "v6_weight": r["v6_weight"],
        "mit_b4_weight": r["mit_b4_weight"],
        "calibration_score": r["calibration_score"]
    }
    for r in search_rows
]).sort_values("calibration_score", ascending=False).reset_index(drop=True)

print("TOP FOUR-MODEL BLEND CANDIDATES")
display(blend_search_df.head(12).round(6))
print("\nSelected independent weights:")
for name, weight in zip(["V4", "V5", "V6", "MiT-B4"], SELECTED_WEIGHTS):
    print(f"{name}: {weight:.3f}")
print("Weights sum:", sum(SELECTED_WEIGHTS))
print("Selected calibration thresholds:", CALIBRATION_THRESHOLDS)
print("V7 weight-anchor calibration score:", round(V7_WEIGHT_ANCHOR_CALIBRATION_SCORE, 6))
print("Three-model anchor calibration score:", round(THREE_MODEL_ANCHOR_CALIBRATION_SCORE, 6))


In [ ]:
# Build a conservative class-aware weight matrix from the already-computed calibration search.
# This reuses search_rows: no additional model inference or probability-cache generation is needed.
# The V12 global blend remains the fallback for every class.

GLOBAL_V12_WEIGHTS = np.asarray(SELECTED_WEIGHTS, dtype=np.float64)
MAX_CLASS_L1_DISTANCE = 0.50
MIN_CLASS_CALIBRATION_GAIN = {
    "defect_a": 0.00025,
    "defect_b": 0.00075,  # More conservative: this class has only 63 positives in all validation.
    "defect_c": 0.00025,
    "defect_d": 0.00025,
}

CLASS_WEIGHT_MATRIX = np.tile(GLOBAL_V12_WEIGHTS, (N_CLASSES, 1))
CLASS_WEIGHT_SELECTION = {}

for class_idx, class_name in enumerate(CLASSES):
    baseline_contribution = float(
        best_calibration_row["per_class"][class_name]["global_score_contribution"]
    )
    eligible_rows = []
    for row in search_rows:
        candidate_weights = np.asarray(row["weights"], dtype=np.float64)
        distance = float(np.abs(candidate_weights - GLOBAL_V12_WEIGHTS).sum())
        if distance <= MAX_CLASS_L1_DISTANCE + 1e-9:
            eligible_rows.append((row, distance))

    if not eligible_rows:
        raise RuntimeError(f"No conservative candidate weights found for {class_name}.")

    best_row, best_distance = max(
        eligible_rows,
        key=lambda item: float(item[0]["per_class"][class_name]["global_score_contribution"])
    )
    best_contribution = float(
        best_row["per_class"][class_name]["global_score_contribution"]
    )
    gain = best_contribution - baseline_contribution
    required_gain = MIN_CLASS_CALIBRATION_GAIN[class_name]

    if gain >= required_gain:
        CLASS_WEIGHT_MATRIX[class_idx] = np.asarray(best_row["weights"], dtype=np.float64)
        decision = "class-specific candidate accepted"
        selected_stage = best_row["stage"]
    else:
        decision = "retain V12 global weights (gain below conservative minimum)"
        selected_stage = "V12_global_fallback"
        best_distance = 0.0
        best_contribution = baseline_contribution
        gain = 0.0

    CLASS_WEIGHT_SELECTION[class_name] = {
        "decision": decision,
        "selected_stage": selected_stage,
        "weights": CLASS_WEIGHT_MATRIX[class_idx].tolist(),
        "calibration_contribution_gain": float(gain),
        "required_minimum_gain": float(required_gain),
        "l1_distance_from_v12_global": float(best_distance),
    }

CLASS_WEIGHT_MATRIX = validate_blend_weight_matrix(CLASS_WEIGHT_MATRIX).astype(np.float32)

# Re-optimise thresholds against the chosen per-class weight rows on CAL_IDX.
CLASSWISE_CALIBRATION_SCORE, CLASSWISE_CALIBRATION_THRESHOLDS, CLASSWISE_CALIBRATION_DETAILS = (
    optimise_thresholds_for_weights(CLASS_WEIGHT_MATRIX, CAL_IDX)
)

print("CLASS-AWARE WEIGHT SEARCH")
print("=" * 65)
print("V12 global reference:", GLOBAL_V12_WEIGHTS.tolist())
print("Conservative max L1 distance:", MAX_CLASS_L1_DISTANCE)
print("Class-aware calibration score:", round(CLASSWISE_CALIBRATION_SCORE, 6))
print("V12 global calibration score:", round(best_calibration_row["calibration_score"], 6))
print("Calibration score delta:", round(CLASSWISE_CALIBRATION_SCORE - best_calibration_row["calibration_score"], 6))
print("\nWeights by class [V4, V5, V6, MiT-B4]:")
for class_idx, class_name in enumerate(CLASSES):
    print(f"{class_name}: {CLASS_WEIGHT_MATRIX[class_idx].round(3).tolist()} | {CLASS_WEIGHT_SELECTION[class_name]['decision']}")
print("\nCalibration thresholds:", CLASSWISE_CALIBRATION_THRESHOLDS)

if not np.allclose(CLASS_WEIGHT_MATRIX.sum(axis=1), 1.0, atol=1e-6):
    raise RuntimeError("At least one selected class-specific weight row does not sum to one.")


In [ ]:
# Compare the V13 class-aware baseline with an expanded Colab V7 hybrid.
# All parameter selection happens on CAL_IDX. HOLDOUT_IDX is diagnostic only.
# The official metric has GLOBAL denominators across all image-class pairs.

HYBRID_BETA_GRID = np.round(np.arange(0.00, 1.001, 0.10), 2).astype(np.float32)
HYBRID_GATE_GRID = [None] + np.round(np.arange(0.05, 0.951, 0.05), 2).tolist()
HYBRID_MIN_GAIN = {
    "defect_a": 0.00040,
    "defect_b": 0.00100,  # Rare class: keep the acceptance rule conservative.
    "defect_c": 0.00040,
    "defect_d": 0.00040,
}
LOGIT_LOCAL_BETA_RADIUS = 0.20
LOGIT_LOCAL_GATE_RADIUS = 0.10


def _fuse_hybrid_probabilities(base_probs, colab_probs, beta, fusion_mode):
    """Combine probability maps by arithmetic probability pooling or logit pooling."""
    beta = float(beta)
    if fusion_mode == "probability":
        return (1.0 - beta) * base_probs + beta * colab_probs
    if fusion_mode == "logit":
        eps = 1e-4
        base_clipped = np.clip(np.asarray(base_probs, dtype=np.float32), eps, 1.0 - eps)
        colab_clipped = np.clip(np.asarray(colab_probs, dtype=np.float32), eps, 1.0 - eps)
        base_logits = np.log(base_clipped) - np.log1p(-base_clipped)
        colab_logits = np.log(colab_clipped) - np.log1p(-colab_clipped)
        mixed_logits = (1.0 - beta) * base_logits + beta * colab_logits
        return (1.0 / (1.0 + np.exp(-mixed_logits))).astype(np.float32, copy=False)
    raise ValueError(f"Unknown fusion mode: {fusion_mode}")


def _hybrid_global_contribution(
    dice_sum, empty_fp_pixels, correct_empty, class_empty_pairs,
    total_positive, total_empty
):
    """Additive per-class contribution to the GLOBAL competition score.

    The 0.20 false-positive baseline is allocated across classes in proportion
    to each class's empty-pair count. It must not be added once per class.
    Summing this function over all classes reproduces the official score.
    """
    total_empty_pixels = int(total_empty) * N_PIXELS
    class_empty_pixels = int(class_empty_pairs) * N_PIXELS
    return (
        0.70 * float(dice_sum) / max(int(total_positive), 1)
        + 0.20 * (class_empty_pixels - int(empty_fp_pixels)) / max(total_empty_pixels, 1)
        + 0.10 * float(correct_empty) / max(int(total_empty), 1)
    )


def _build_hybrid_histograms(class_idx, indices, beta, fusion_mode="probability"):
    indices = np.asarray(indices, dtype=np.int64)
    n_rows = len(indices)
    hist_all = np.zeros((n_rows, N_THRESHOLD_BINS), dtype=np.uint32)
    hist_positive = np.zeros_like(hist_all)
    class_gt = gt_counts[indices, class_idx]
    targets_positive = class_gt > 0

    for chunk_start in range(0, n_rows, CAL_CHUNK_SIZE):
        chunk_end = min(chunk_start + CAL_CHUNK_SIZE, n_rows)
        row_indices = indices[chunk_start:chunk_end]
        base_probs = combine_cached_probabilities(row_indices, class_idx, CLASS_WEIGHT_MATRIX)
        if COLAB_V7_AVAILABLE:
            colab_probs = np.asarray(val_colab_seg_probs[row_indices, class_idx], dtype=np.float32)
            combined = _fuse_hybrid_probabilities(base_probs, colab_probs, beta, fusion_mode)
        else:
            combined = base_probs
        targets = np.asarray(val_masks[row_indices, class_idx] > 0)
        combined_flat = combined.reshape(len(row_indices), -1)
        targets_flat = targets.reshape(len(row_indices), -1)
        bins = np.searchsorted(THRESHOLD_GRID, combined_flat, side="right").astype(np.uint8)
        for local_row in range(len(row_indices)):
            hist_all[chunk_start + local_row] = np.bincount(
                bins[local_row], minlength=N_THRESHOLD_BINS
            ).astype(np.uint32)
            hist_positive[chunk_start + local_row] = np.bincount(
                bins[local_row][targets_flat[local_row]], minlength=N_THRESHOLD_BINS
            ).astype(np.uint32)

    suffix_all = np.cumsum(hist_all[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
    suffix_positive = np.cumsum(hist_positive[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
    return suffix_all, suffix_positive, class_gt


def _search_one_class_hybrid(
    class_idx, indices, beta_candidates, gate_candidates, fusion_mode="probability"
):
    indices = np.asarray(indices, dtype=np.int64)
    total_positive = int((gt_counts[indices] > 0).sum())
    total_empty = int((gt_counts[indices] == 0).sum())
    if total_positive == 0 or total_empty == 0:
        raise ValueError("Calibration split must include positive and empty pairs for every class.")

    class_name = CLASSES[class_idx]
    classifier_scores = (
        np.asarray(val_colab_cls_probs[indices, class_idx], dtype=np.float32)
        if COLAB_V7_AVAILABLE else None
    )

    best = None
    baseline = None
    for beta in beta_candidates:
        suffix_all, suffix_positive, class_gt = _build_hybrid_histograms(
            class_idx, indices, beta, fusion_mode=fusion_mode
        )
        positive = class_gt > 0
        empty = ~positive
        class_empty_pairs = int(empty.sum())

        for gate_threshold in gate_candidates:
            if gate_threshold is None or not COLAB_V7_AVAILABLE:
                gate_pass = np.ones(len(indices), dtype=bool)
            else:
                gate_pass = classifier_scores >= float(gate_threshold)

            for threshold_index, threshold in enumerate(THRESHOLD_GRID):
                predicted_counts = suffix_all[:, threshold_index + 1]
                intersection_counts = suffix_positive[:, threshold_index + 1]
                gated_predicted = np.where(gate_pass, predicted_counts, 0)
                gated_intersection = np.where(gate_pass, intersection_counts, 0)

                if positive.any():
                    pair_dice = (
                        2.0 * gated_intersection[positive]
                        / np.maximum(gated_predicted[positive] + class_gt[positive], 1)
                    )
                    dice_sum = float(pair_dice.sum(dtype=np.float64))
                else:
                    dice_sum = 0.0

                empty_fp_pixels = int(gated_predicted[empty].sum(dtype=np.uint64))
                correct_empty = int((empty & ((~gate_pass) | (predicted_counts == 0))).sum())
                contribution = _hybrid_global_contribution(
                    dice_sum, empty_fp_pixels, correct_empty, class_empty_pairs,
                    total_positive, total_empty
                )
                row = {
                    "class": class_name,
                    "fusion_mode": fusion_mode,
                    "beta_colab_v7": float(beta),
                    "gate_threshold": None if gate_threshold is None else float(gate_threshold),
                    "mask_threshold": float(threshold),
                    "global_score_contribution": float(contribution),
                    "foreground_dice_class": float(dice_sum / max(int(positive.sum()), 1)),
                    "empty_pairs_class": class_empty_pairs,
                    "positive_pairs_class": int(positive.sum()),
                }
                if best is None or row["global_score_contribution"] > best["global_score_contribution"]:
                    best = row
                if (
                    fusion_mode == "probability" and float(beta) == 0.0
                    and gate_threshold is None
                ):
                    if baseline is None or row["global_score_contribution"] > baseline["global_score_contribution"]:
                        baseline = row
        del suffix_all, suffix_positive
        gc.collect()

    return best, baseline


def _local_logit_search_candidates(probability_best):
    """Small local logit-space search around the best probability-space candidate."""
    center_beta = float(probability_best["beta_colab_v7"])
    beta_values = sorted(set(float(np.clip(center_beta + delta, 0.0, 1.0))
                             for delta in (-LOGIT_LOCAL_BETA_RADIUS, -0.10, 0.0, 0.10, LOGIT_LOCAL_BETA_RADIUS)))
    center_gate = probability_best["gate_threshold"]
    if center_gate is None:
        gate_center = 0.50
        gate_values = [None] + [float(np.clip(gate_center + d, 0.05, 0.95))
                                for d in (-LOGIT_LOCAL_GATE_RADIUS, 0.0, LOGIT_LOCAL_GATE_RADIUS)]
    else:
        gate_center = float(center_gate)
        gate_values = [None] + [float(np.clip(gate_center + d, 0.05, 0.95))
                                for d in (-LOGIT_LOCAL_GATE_RADIUS, -0.05, 0.0, 0.05, LOGIT_LOCAL_GATE_RADIUS)]
    gate_values = list(dict.fromkeys(gate_values))
    return beta_values, gate_values


def optimise_fixed_hybrid_thresholds(beta_by_class, gate_by_class, fusion_by_class, indices):
    """Refit pixel thresholds for fixed class beta/gate/fusion choices."""
    selected_thresholds = {}
    per_class = {}
    total_score = 0.0
    for class_idx, class_name in enumerate(CLASSES):
        beta = float(beta_by_class[class_name])
        gate = gate_by_class[class_name]
        fusion_mode = fusion_by_class[class_name]
        best, _ = _search_one_class_hybrid(
            class_idx, indices, [beta], [gate], fusion_mode=fusion_mode
        )
        selected_thresholds[class_name] = float(best["mask_threshold"])
        per_class[class_name] = best
        total_score += float(best["global_score_contribution"])
    return float(total_score), selected_thresholds, per_class


def remove_small_components(mask, min_area):
    """Remove 8-connected foreground components smaller than min_area pixels."""
    mask = np.asarray(mask, dtype=bool)
    min_area = int(min_area)
    if min_area <= 1 or not mask.any():
        return mask
    labels, _ = ndimage.label(mask, structure=np.ones((3, 3), dtype=np.uint8))
    component_sizes = np.bincount(labels.ravel())
    keep = component_sizes >= min_area
    keep[0] = False
    return keep[labels]


def calibrate_component_min_areas(beta_by_class, gate_by_class, fusion_by_class, thresholds, indices):
    """Tune small-component removal on CAL_IDX after blend/gate/pixel threshold are fixed."""
    indices = np.asarray(indices, dtype=np.int64)
    total_positive = int((gt_counts[indices] > 0).sum())
    total_empty = int((gt_counts[indices] == 0).sum())
    total_empty_pixels = total_empty * N_PIXELS
    area_candidates = [0, 2, 4, 8, 16, 24, 32, 48, 64, 96, 128, 192]
    min_areas = {}
    rows = []
    per_class_details = {}
    for class_idx, class_name in enumerate(CLASSES):
        beta = float(beta_by_class[class_name])
        gate = gate_by_class[class_name]
        fusion_mode = fusion_by_class[class_name]
        threshold = float(thresholds[class_name])
        records = []
        for start in range(0, len(indices), CAL_CHUNK_SIZE):
            rows_idx = indices[start:start + CAL_CHUNK_SIZE]
            base = combine_cached_probabilities(rows_idx, class_idx, CLASS_WEIGHT_MATRIX)
            if COLAB_V7_AVAILABLE:
                aux = np.asarray(val_colab_seg_probs[rows_idx, class_idx], dtype=np.float32)
                combined = _fuse_hybrid_probabilities(base, aux, beta, fusion_mode)
                if gate is None:
                    gate_pass = np.ones(len(rows_idx), dtype=bool)
                else:
                    q = np.asarray(val_colab_cls_probs[rows_idx, class_idx], dtype=np.float32)
                    gate_pass = q >= float(gate)
            else:
                combined = base
                gate_pass = np.ones(len(rows_idx), dtype=bool)
            pred = (combined >= threshold) & gate_pass[:, None, None]
            target = np.asarray(val_masks[rows_idx, class_idx] > 0)
            target_counts = np.count_nonzero(target, axis=(1, 2))
            for local_i in range(len(rows_idx)):
                labels, _ = ndimage.label(pred[local_i], structure=np.ones((3, 3), dtype=np.uint8))
                sizes = np.bincount(labels.ravel())[1:]
                intersections = np.bincount(
                    labels.ravel(), weights=target[local_i].ravel().astype(np.float32)
                )[1:]
                records.append((int(target_counts[local_i]), sizes.astype(np.int64), intersections.astype(np.float64)))

        best_row = None
        baseline_row = None
        for min_area in area_candidates:
            dice_sum = 0.0
            fp_pixels = 0
            correct_empty = 0
            class_positive = 0
            class_empty = 0
            for target_count, sizes, intersections in records:
                keep = sizes >= int(min_area)
                pred_count = int(sizes[keep].sum()) if keep.any() else 0
                intersection = float(intersections[keep].sum()) if keep.any() else 0.0
                if target_count > 0:
                    dice_sum += 2.0 * intersection / max(pred_count + target_count, 1)
                    class_positive += 1
                else:
                    class_empty += 1
                    fp_pixels += pred_count
                    correct_empty += int(pred_count == 0)
            # Additive class contribution excluding the shared +0.20 FP baseline.
            contribution = (
                0.70 * dice_sum / max(total_positive, 1)
                - 0.20 * fp_pixels / max(total_empty_pixels, 1)
                + 0.10 * correct_empty / max(total_empty, 1)
            )
            row = {
                "class": class_name, "min_component_area": int(min_area),
                "global_score_contribution_excluding_shared_fp_baseline": float(contribution),
                "foreground_dice_class": float(dice_sum / max(class_positive, 1)),
                "empty_pairs_class": int(class_empty), "positive_pairs_class": int(class_positive),
                "empty_fp_pixels": int(fp_pixels), "correct_empty_pairs": int(correct_empty),
            }
            rows.append(row)
            if best_row is None or contribution > best_row["global_score_contribution_excluding_shared_fp_baseline"]:
                best_row = row
            if int(min_area) == 0:
                baseline_row = row
        if baseline_row is None or best_row is None:
            raise RuntimeError(f"Component-area search failed for {class_name}.")
        gain = float(best_row["global_score_contribution_excluding_shared_fp_baseline"]
                     - baseline_row["global_score_contribution_excluding_shared_fp_baseline"])
        # Require a small but non-trivial CAL gain before enabling a post-processing rule.
        chosen_area = int(best_row["min_component_area"]) if gain >= 0.00005 else 0
        min_areas[class_name] = chosen_area
        per_class_details[class_name] = {
            "selected_min_component_area": chosen_area,
            "best_candidate_min_component_area": int(best_row["min_component_area"]),
            "calibration_gain": gain,
        }
    return min_areas, pd.DataFrame(rows), per_class_details


def evaluate_hybrid_config(
    beta_by_class, gate_by_class, fusion_by_class, thresholds, indices, component_min_areas=None
):
    """Direct official-metric evaluator; denominators are global across all classes."""
    indices = np.asarray(indices, dtype=np.int64)
    positive_dice_sum = 0.0
    positive_pairs = 0
    empty_fp_pixels = 0
    empty_pixels = 0
    correct_empty_pairs = 0
    empty_pairs = 0
    class_metrics = {}

    for class_idx, class_name in enumerate(CLASSES):
        class_dice_sum = 0.0
        class_positive_pairs = 0
        class_empty_fp = 0
        class_empty_pixel_count = 0
        class_correct_empty = 0
        class_empty_pairs = 0
        beta = float(beta_by_class[class_name])
        gate = gate_by_class[class_name]
        fusion_mode = fusion_by_class[class_name]
        threshold = float(thresholds[class_name])

        for start in range(0, len(indices), CAL_CHUNK_SIZE):
            rows = indices[start:start + CAL_CHUNK_SIZE]
            base = combine_cached_probabilities(rows, class_idx, CLASS_WEIGHT_MATRIX)
            if COLAB_V7_AVAILABLE:
                colab = np.asarray(val_colab_seg_probs[rows, class_idx], dtype=np.float32)
                combined = _fuse_hybrid_probabilities(base, colab, beta, fusion_mode)
                if gate is not None:
                    q = np.asarray(val_colab_cls_probs[rows, class_idx], dtype=np.float32)
                    gate_pass = q >= float(gate)
                else:
                    gate_pass = np.ones(len(rows), dtype=bool)
            else:
                combined = base
                gate_pass = np.ones(len(rows), dtype=bool)
            prediction = (combined >= threshold) & gate_pass[:, None, None]
            min_component_area = 0 if component_min_areas is None else int(component_min_areas.get(class_name, 0))
            if min_component_area > 1:
                prediction = np.stack([
                    remove_small_components(row_mask, min_component_area) for row_mask in prediction
                ], axis=0)
            target = np.asarray(val_masks[rows, class_idx] > 0)
            target_count = np.count_nonzero(target, axis=(1, 2))
            pred_count = np.count_nonzero(prediction, axis=(1, 2))
            intersection = np.count_nonzero(prediction & target, axis=(1, 2))
            positive = target_count > 0
            empty = ~positive
            if positive.any():
                pair_dice = 2.0 * intersection[positive] / np.maximum(
                    pred_count[positive] + target_count[positive], 1
                )
                class_dice_sum += float(pair_dice.sum())
                class_positive_pairs += int(positive.sum())
            if empty.any():
                class_empty_pairs += int(empty.sum())
                class_empty_fp += int(pred_count[empty].sum())
                class_empty_pixel_count += int(empty.sum()) * N_PIXELS
                class_correct_empty += int((pred_count[empty] == 0).sum())

        positive_dice_sum += class_dice_sum
        positive_pairs += class_positive_pairs
        empty_fp_pixels += class_empty_fp
        empty_pixels += class_empty_pixel_count
        correct_empty_pairs += class_correct_empty
        empty_pairs += class_empty_pairs
        class_metrics[class_name] = {
            "positive_pairs": class_positive_pairs,
            "empty_pairs": class_empty_pairs,
            "foreground_dice": class_dice_sum / max(class_positive_pairs, 1),
            "false_positive_score": 1.0 - class_empty_fp / max(class_empty_pixel_count, 1),
            "empty_accuracy": class_correct_empty / max(class_empty_pairs, 1),
            "beta_colab_v7": beta,
            "gate_threshold": gate,
            "fusion_mode": fusion_mode,
            "mask_threshold": threshold,
        }

    foreground_dice = positive_dice_sum / max(positive_pairs, 1)
    false_positive_score = 1.0 - empty_fp_pixels / max(empty_pixels, 1)
    empty_accuracy = correct_empty_pairs / max(empty_pairs, 1)
    score = 0.70 * foreground_dice + 0.20 * false_positive_score + 0.10 * empty_accuracy
    return {
        "foreground_dice": float(foreground_dice),
        "false_positive_score": float(false_positive_score),
        "empty_accuracy": float(empty_accuracy),
        "weighted_score": float(score),
        "positive_pairs": int(positive_pairs),
        "empty_pairs": int(empty_pairs),
        "class_metrics": class_metrics,
    }


if COLAB_V7_AVAILABLE:
    print("CALIBRATING THE EXPANDED COLAB V7 HYBRID ON CAL_IDX ONLY")
    print("Probability beta candidates:", HYBRID_BETA_GRID.tolist())
    print("Classifier gate candidates:", HYBRID_GATE_GRID)
    print("Logit mode: local refinement around each class's best probability blend")
    CLASS_BETA = {}
    CLASS_GATE_THRESHOLDS = {}
    CLASS_FUSION_MODE = {}
    HYBRID_CAL_THRESHOLDS = {}
    HYBRID_SELECTION_ROWS = []
    HYBRID_CLASS_SELECTION = {}

    for class_idx, class_name in enumerate(CLASSES):
        best_probability, baseline_row = _search_one_class_hybrid(
            class_idx, CAL_IDX, HYBRID_BETA_GRID.tolist(), HYBRID_GATE_GRID,
            fusion_mode="probability"
        )
        if baseline_row is None:
            raise RuntimeError(f"Exact no-Colab baseline was not evaluated for {class_name}.")

        logit_beta_candidates, logit_gate_candidates = _local_logit_search_candidates(best_probability)
        best_logit, _ = _search_one_class_hybrid(
            class_idx, CAL_IDX, logit_beta_candidates, logit_gate_candidates,
            fusion_mode="logit"
        )
        best_row = max(
            [best_probability, best_logit],
            key=lambda row: float(row["global_score_contribution"])
        )
        gain = float(best_row["global_score_contribution"] - baseline_row["global_score_contribution"])
        min_gain = float(HYBRID_MIN_GAIN[class_name])
        accepted = gain >= min_gain
        chosen = best_row if accepted else baseline_row

        CLASS_BETA[class_name] = float(chosen["beta_colab_v7"])
        CLASS_GATE_THRESHOLDS[class_name] = chosen["gate_threshold"]
        CLASS_FUSION_MODE[class_name] = str(chosen["fusion_mode"])
        HYBRID_CAL_THRESHOLDS[class_name] = float(chosen["mask_threshold"])
        HYBRID_CLASS_SELECTION[class_name] = {
            "accepted": bool(accepted),
            "decision": "Colab V7 contribution accepted" if accepted else "retain V13 class-aware baseline",
            "baseline_calibration_contribution": float(baseline_row["global_score_contribution"]),
            "best_probability_candidate": best_probability,
            "best_local_logit_candidate": best_logit,
            "best_candidate_calibration_contribution": float(best_row["global_score_contribution"]),
            "observed_gain": gain,
            "required_gain": min_gain,
            "selected_beta_colab_v7": CLASS_BETA[class_name],
            "selected_gate_threshold": CLASS_GATE_THRESHOLDS[class_name],
            "selected_fusion_mode": CLASS_FUSION_MODE[class_name],
            "selected_calibration_mask_threshold": HYBRID_CAL_THRESHOLDS[class_name],
        }
        HYBRID_SELECTION_ROWS.append({
            **best_row,
            "baseline_contribution": baseline_row["global_score_contribution"],
            "gain": gain,
            "accepted": accepted,
            "selected": bool(chosen is best_row),
        })
        print(
            f"{class_name}: fusion={CLASS_FUSION_MODE[class_name]}, "
            f"beta={CLASS_BETA[class_name]:.2f}, gate={CLASS_GATE_THRESHOLDS[class_name]}, "
            f"mask_t={HYBRID_CAL_THRESHOLDS[class_name]:.3f}, "
            f"gain={gain:+.6f}, accepted={accepted}"
        )

    hybrid_calibration_score = float(sum(
        (selection["best_candidate_calibration_contribution"]
         if selection["accepted"] else selection["baseline_calibration_contribution"])
        for selection in HYBRID_CLASS_SELECTION.values()
    ))

    # Refit pixel thresholds on all validation images only after selecting beta/gates/fusion on CAL.
    full_validation_score, FINAL_THRESHOLDS, final_threshold_details = optimise_fixed_hybrid_thresholds(
        CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE, ALL_VAL_IDX
    )
    hybrid_calibration_thresholds = dict(HYBRID_CAL_THRESHOLDS)
else:
    CLASS_BETA = {name: 0.0 for name in CLASSES}
    CLASS_GATE_THRESHOLDS = {name: None for name in CLASSES}
    CLASS_FUSION_MODE = {name: "probability" for name in CLASSES}
    hybrid_calibration_score = float(CLASSWISE_CALIBRATION_SCORE)
    hybrid_calibration_thresholds = dict(CLASSWISE_CALIBRATION_THRESHOLDS)
    full_validation_score, FINAL_THRESHOLDS, final_threshold_details = optimise_thresholds_for_weights(
        CLASS_WEIGHT_MATRIX, ALL_VAL_IDX
    )
    HYBRID_CLASS_SELECTION = {
        name: {"accepted": False, "decision": "Colab V7 checkpoints missing; V13 fallback",
               "selected_beta_colab_v7": 0.0, "selected_gate_threshold": None,
               "selected_fusion_mode": "probability"}
        for name in CLASSES
    }
    HYBRID_SELECTION_ROWS = []

# Small-component removal is a post-processing experiment, calibrated only on CAL_IDX.
# It is applied only when it gives a minimum 0.00005 additive CAL contribution gain.
COMPONENT_MIN_AREAS, component_area_search_df, COMPONENT_AREA_SELECTION = calibrate_component_min_areas(
    CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
    hybrid_calibration_thresholds, CAL_IDX
)
component_calibration_metrics = evaluate_hybrid_config(
    CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
    hybrid_calibration_thresholds, CAL_IDX, component_min_areas=COMPONENT_MIN_AREAS
)
component_area_search_df.to_csv(OUTPUT_DIR / "component_min_area_search.csv", index=False)
print("Selected minimum component areas:", COMPONENT_MIN_AREAS)
print("Component-filtered CAL score:", round(component_calibration_metrics["weighted_score"], 6))

# V19 does not apply the calibrated component filter to final masks. In the V16
# run, it improved CAL but reduced the reserved HOLDOUT score (0.721759 with
# filtering vs 0.722160 without filtering). Keep the candidate areas for audit.
COMPONENT_MIN_AREAS_CALIBRATED = dict(COMPONENT_MIN_AREAS)
COMPONENT_MIN_AREAS = {class_name: 0 for class_name in CLASSES}
print("V19 final component filter (disabled):", COMPONENT_MIN_AREAS)


COLAB_V7_BLEND_ACTIVE = bool(
    COLAB_V7_AVAILABLE and any(
        float(CLASS_BETA[name]) > 0.0 or CLASS_GATE_THRESHOLDS[name] is not None
        for name in CLASSES
    )
)
print("Colab V7 branch selected for final test inference:", COLAB_V7_BLEND_ACTIVE)

# Direct metric-consistency checks: class contributions must add up to the global scorer.
if COLAB_V7_AVAILABLE:
    hybrid_calibration_metrics = evaluate_hybrid_config(
        CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
        hybrid_calibration_thresholds, CAL_IDX
    )
    if not np.isclose(
        hybrid_calibration_score, hybrid_calibration_metrics["weighted_score"],
        atol=2e-5, rtol=0.0
    ):
        raise RuntimeError(
            "Hybrid CAL contribution sum does not match the direct official evaluator: "
            f"{hybrid_calibration_score:.8f} vs {hybrid_calibration_metrics['weighted_score']:.8f}"
        )
    full_validation_metrics = evaluate_hybrid_config(
        CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
        FINAL_THRESHOLDS, ALL_VAL_IDX
    )
    full_validation_metrics_with_components = evaluate_hybrid_config(
        CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
        FINAL_THRESHOLDS, ALL_VAL_IDX, component_min_areas=COMPONENT_MIN_AREAS_CALIBRATED
    )
    if not np.isclose(
        full_validation_score, full_validation_metrics["weighted_score"],
        atol=2e-5, rtol=0.0
    ):
        raise RuntimeError(
            "Full-validation contribution sum does not match the direct official evaluator: "
            f"{full_validation_score:.8f} vs {full_validation_metrics['weighted_score']:.8f}"
        )
else:
    hybrid_calibration_metrics = None
    full_validation_metrics = None
    full_validation_metrics_with_components = None

# Evaluate the selected candidate and the V13 baseline on the same reserved diagnostic split.
hybrid_holdout_metrics_unfiltered = evaluate_hybrid_config(
    CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
    hybrid_calibration_thresholds, HOLDOUT_IDX
) if COLAB_V7_AVAILABLE else evaluate_mix_on_indices(
    CLASS_WEIGHT_MATRIX, CLASSWISE_CALIBRATION_THRESHOLDS, HOLDOUT_IDX
)
hybrid_holdout_metrics_component_candidate = evaluate_hybrid_config(
    CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
    hybrid_calibration_thresholds, HOLDOUT_IDX,
    component_min_areas=COMPONENT_MIN_AREAS_CALIBRATED
) if COLAB_V7_AVAILABLE else hybrid_holdout_metrics_unfiltered
hybrid_holdout_metrics = evaluate_hybrid_config(
    CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
    hybrid_calibration_thresholds, HOLDOUT_IDX, component_min_areas=COMPONENT_MIN_AREAS
) if COLAB_V7_AVAILABLE else hybrid_holdout_metrics_unfiltered
v13_baseline_holdout_metrics = evaluate_mix_on_indices(
    CLASS_WEIGHT_MATRIX, CLASSWISE_CALIBRATION_THRESHOLDS, HOLDOUT_IDX
)

print("\nHYBRID HOLDOUT COMPARISON")
print("=" * 65)
print("V13 baseline holdout:", round(v13_baseline_holdout_metrics["weighted_score"], 6))
print("Hybrid candidate holdout:", round(hybrid_holdout_metrics["weighted_score"], 6))
print("Component candidate holdout:", round(hybrid_holdout_metrics_component_candidate["weighted_score"], 6))
print("Hybrid minus V13:", round(
    hybrid_holdout_metrics["weighted_score"] - v13_baseline_holdout_metrics["weighted_score"], 6
))
print("Selected beta by class:", CLASS_BETA)
print("Selected gate thresholds:", CLASS_GATE_THRESHOLDS)
print("Selected fusion modes:", CLASS_FUSION_MODE)
print("Final mask thresholds:", FINAL_THRESHOLDS)
print("CAL official score:", round(hybrid_calibration_score, 6))
print("Full-validation in-sample score (without component filter):", round(full_validation_score, 6))
if full_validation_metrics_with_components is not None:
    print("Full-validation in-sample score (with component filter):", round(full_validation_metrics_with_components["weighted_score"], 6))
if not COLAB_V7_AVAILABLE:
    print("\nFALLBACK NOTICE: this run did not include Colab V7 checkpoints; it is a V13-equivalent fallback.")
elif all(float(v) == 0.0 and CLASS_GATE_THRESHOLDS[k] is None for k, v in CLASS_BETA.items()):
    print("\nThe calibration search retained the V13 baseline for all classes; no Colab contribution was justified.")

FINAL_BLEND_CONFIG = {
    "notebook_version": "v19_adaptive_soft_gate_conflict_aware_hybrid",
    "baseline_public_score": BEST_PUBLIC_SCORE_BASELINE,
    "public_score_goal": PUBLIC_SCORE_TARGET,
    "models": ["V4 U-Net/ResNet34", "V5 SegFormer-B0", "V6 DeepLabV3+/ResNet50", "MiT-B4 SegFormer"],
    "baseline_class_aware_weights": {
        class_name: {
            model_name: float(CLASS_WEIGHT_MATRIX[class_idx, model_idx])
            for model_idx, model_name in enumerate(["v4", "v5", "v6", "mit_b4"])
        }
        for class_idx, class_name in enumerate(CLASSES)
    },
    "optional_colab_v7_models_available": bool(COLAB_V7_AVAILABLE),
    "colab_v7_checkpoint_paths": {k: str(v) if v is not None else None for k, v in COLAB_V7_PATHS.items()},
    "colab_v7_group_weights": COLAB_V7_GROUP_WEIGHTS,
    "colab_v7_beta_by_class": CLASS_BETA,
    "colab_v7_classifier_gate_by_class": CLASS_GATE_THRESHOLDS,
    "colab_v7_fusion_mode_by_class": CLASS_FUSION_MODE,
    "component_min_area_by_class": COMPONENT_MIN_AREAS,
    "calibrated_component_candidate_by_class": COMPONENT_MIN_AREAS_CALIBRATED,
    "component_candidate_holdout_metrics": hybrid_holdout_metrics_component_candidate,
    "component_area_selection": COMPONENT_AREA_SELECTION,
    "component_filtered_calibration_metrics": component_calibration_metrics,
    "hybrid_holdout_metrics_without_component_filter": hybrid_holdout_metrics_unfiltered,
    "calibration_mask_thresholds": hybrid_calibration_thresholds,
    "final_mask_thresholds": FINAL_THRESHOLDS,
    "hybrid_class_selection": HYBRID_CLASS_SELECTION,
    "hybrid_calibration_score": float(hybrid_calibration_score),
    "hybrid_calibration_metrics": hybrid_calibration_metrics,
    "full_validation_calibrated_objective": float(full_validation_score),
    "full_validation_metrics": full_validation_metrics,
    "full_validation_metrics_with_components": full_validation_metrics_with_components,
    "hybrid_holdout_metrics": hybrid_holdout_metrics,
    "v13_baseline_holdout_metrics": v13_baseline_holdout_metrics,
    "beta_search_grid": HYBRID_BETA_GRID.tolist(),
    "gate_search_grid": HYBRID_GATE_GRID,
    "pixel_threshold_step": 0.005,
    "logit_search": "local neighbourhood around best probability-space candidate",
    "inference_mode": MAIN_INFERENCE_MODE,
    "mit_b4_inference_mode": MIT_INFERENCE_MODE,
    "metric": "0.70 foreground Dice + 0.20 false-positive score + 0.10 empty-prediction accuracy",
    "metric_uses_global_pair_denominators": True,
    "metric_contribution_sum_assertion": "passed" if COLAB_V7_AVAILABLE else "not_applicable_fallback",
    "training_performed": False,
    "holdout_caveat": "All checkpoints were selected/trained using the original validation process; the internal holdout is diagnostic, not a fully untouched external estimate.",
    "public_score_caveat": "No local metric guarantees a higher Kaggle public score. Preserve the previous 0.64701 submission as the baseline."
}

CONFIG_PATH = OUTPUT_DIR / "selected_v19_hybrid_blend.json"
with open(CONFIG_PATH, "w") as f:
    json.dump(FINAL_BLEND_CONFIG, f, indent=2)

blend_search_df.to_csv(OUTPUT_DIR / "global_four_model_weight_search.csv", index=False)
fine_blend_search_df.to_csv(OUTPUT_DIR / "fine_four_model_weight_search.csv", index=False)
pd.DataFrame(CLASS_WEIGHT_SELECTION).T.to_csv(OUTPUT_DIR / "class_aware_weight_selection.csv")
pd.DataFrame(HYBRID_SELECTION_ROWS).to_csv(OUTPUT_DIR / "hybrid_beta_gate_search.csv", index=False)
pd.DataFrame(HYBRID_CLASS_SELECTION).T.to_csv(OUTPUT_DIR / "hybrid_class_selection.csv")
print("\nV19 hybrid configuration saved:", CONFIG_PATH)


In [ ]:
# V19: replace a brittle all-or-none classifier veto with a calibrated soft threshold,
# then test whether overlapping class masks should be arbitrated pixel-by-pixel.
# Selection uses CAL_IDX only. HOLDOUT_IDX is diagnostic and never chooses a policy.

if not COLAB_V7_AVAILABLE:
    raise RuntimeError("V19 requires all eight model checkpoints, including all four Colab V7 auxiliary checkpoints.")

V19_FINAL_THRESHOLDS = {name: float(hybrid_calibration_thresholds[name]) for name in CLASSES}
V19_SOFT_FACTORS = [1.0, 1.5, 2.0, 3.0, 5.0]
V19_MIN_SOFT_GAIN = {
    "defect_a": 0.00020,
    "defect_b": 0.00050,  # Rare positive class: higher acceptance margin.
    "defect_c": 0.00020,
    "defect_d": 0.00020,
}
V19_MIN_CONFLICT_GAIN = 0.00010
V19_MAX_GT_OVERLAP_RATE = 0.0020  # Do not force exclusivity if labels overlap materially.


def v19_class_contribution(indices, class_idx, policy):
    """Official-metric contribution for one class using global pair denominators."""
    indices = np.asarray(indices, dtype=np.int64)
    total_positive = int((gt_counts[indices] > 0).sum())
    total_empty = int((gt_counts[indices] == 0).sum())
    if total_positive <= 0 or total_empty <= 0:
        raise ValueError("V19 calibration/evaluation subset must contain positive and empty pairs.")
    class_empty_pairs = int((gt_counts[indices, class_idx] == 0).sum())
    threshold = float(V19_FINAL_THRESHOLDS[CLASSES[class_idx]])
    class_name = CLASSES[class_idx]
    beta = float(CLASS_BETA[class_name])
    fusion_mode = CLASS_FUSION_MODE[class_name]
    policy_mode = policy["mode"]
    dice_sum = 0.0
    empty_fp_pixels = 0
    correct_empty_pairs = 0

    for start in range(0, len(indices), CAL_CHUNK_SIZE):
        rows = indices[start:start + CAL_CHUNK_SIZE]
        base = combine_cached_probabilities(rows, class_idx, CLASS_WEIGHT_MATRIX)
        aux = np.asarray(val_colab_seg_probs[rows, class_idx], dtype=np.float32)
        combined = _fuse_hybrid_probabilities(base, aux, beta, fusion_mode)
        classifier_prob = np.asarray(val_colab_cls_probs[rows, class_idx], dtype=np.float32)

        if policy_mode == "hard":
            gate = policy.get("gate")
            gate_pass = np.ones(len(rows), dtype=bool) if gate is None else classifier_prob >= float(gate)
            prediction = (combined >= threshold) & gate_pass[:, None, None]
        elif policy_mode == "soft_threshold":
            factor = float(policy["factor"])
            effective_threshold = threshold * (factor - (factor - 1.0) * classifier_prob)
            prediction = combined >= effective_threshold[:, None, None]
        else:
            raise ValueError(f"Unknown V19 classifier policy: {policy_mode}")

        target = np.asarray(val_masks[rows, class_idx] > 0)
        target_count = np.count_nonzero(target, axis=(1, 2))
        pred_count = np.count_nonzero(prediction, axis=(1, 2))
        intersection = np.count_nonzero(prediction & target, axis=(1, 2))
        positive = target_count > 0
        empty = ~positive
        if positive.any():
            pair_dice = 2.0 * intersection[positive] / np.maximum(
                pred_count[positive] + target_count[positive], 1
            )
            dice_sum += float(pair_dice.sum(dtype=np.float64))
        if empty.any():
            empty_fp_pixels += int(pred_count[empty].sum(dtype=np.uint64))
            correct_empty_pairs += int((pred_count[empty] == 0).sum())

    return float(
        0.70 * dice_sum / total_positive
        + 0.20 * (class_empty_pairs * N_PIXELS - empty_fp_pixels) / (total_empty * N_PIXELS)
        + 0.10 * correct_empty_pairs / total_empty
    )


def v19_evaluate_policy(indices, policies, resolve_conflicts=False):
    """Direct global evaluator for four class policies and optional pixel-conflict resolution."""
    indices = np.asarray(indices, dtype=np.int64)
    positive_dice_sum = 0.0
    positive_pairs = 0
    empty_fp_pixels = 0
    empty_pixels = 0
    correct_empty_pairs = 0
    empty_pairs = 0
    gt_overlap_pixels = 0
    gt_foreground_pixels = 0
    for start in range(0, len(indices), CAL_CHUNK_SIZE):
        rows = indices[start:start + CAL_CHUNK_SIZE]
        probabilities = []
        thresholds_by_class = []
        gate_pass_by_class = []
        target = np.asarray(val_masks[rows] > 0)
        class_masks = []

        for class_idx, class_name in enumerate(CLASSES):
            base = combine_cached_probabilities(rows, class_idx, CLASS_WEIGHT_MATRIX)
            aux = np.asarray(val_colab_seg_probs[rows, class_idx], dtype=np.float32)
            combined = _fuse_hybrid_probabilities(
                base, aux, float(CLASS_BETA[class_name]), CLASS_FUSION_MODE[class_name]
            )
            q = np.asarray(val_colab_cls_probs[rows, class_idx], dtype=np.float32)
            policy = policies[class_name]
            threshold = float(V19_FINAL_THRESHOLDS[class_name])
            if policy["mode"] == "hard":
                gate = policy.get("gate")
                gate_pass = np.ones(len(rows), dtype=bool) if gate is None else q >= float(gate)
                effective_threshold = np.full(len(rows), threshold, dtype=np.float32)
            elif policy["mode"] == "soft_threshold":
                factor = float(policy["factor"])
                effective_threshold = threshold * (factor - (factor - 1.0) * q)
                gate_pass = np.ones(len(rows), dtype=bool)
            else:
                raise ValueError(f"Unknown V19 classifier policy: {policy['mode']}")
            prediction = (combined >= effective_threshold[:, None, None]) & gate_pass[:, None, None]
            probabilities.append(combined)
            thresholds_by_class.append(effective_threshold)
            gate_pass_by_class.append(gate_pass)
            class_masks.append(prediction)

        masks = np.stack(class_masks, axis=1)
        if resolve_conflicts:
            # Only among pixels already predicted positive by >=1 class.
            normalized_margin = np.stack([
                probabilities[k] / np.maximum(thresholds_by_class[k][:, None, None], 1e-6)
                for k in range(N_CLASSES)
            ], axis=1)
            normalized_margin = np.where(masks, normalized_margin, -np.inf)
            winners = np.argmax(normalized_margin, axis=1)
            class_ids = np.arange(N_CLASSES, dtype=np.int64).reshape(1, N_CLASSES, 1, 1)
            masks = masks & (winners[:, None, :, :] == class_ids)

        target_counts = np.count_nonzero(target, axis=(2, 3))
        pred_counts = np.count_nonzero(masks, axis=(2, 3))
        intersections = np.count_nonzero(masks & target, axis=(2, 3))
        positive = target_counts > 0
        empty = ~positive
        if positive.any():
            pair_dice = 2.0 * intersections[positive] / np.maximum(
                pred_counts[positive] + target_counts[positive], 1
            )
            positive_dice_sum += float(pair_dice.sum(dtype=np.float64))
            positive_pairs += int(positive.sum())
        if empty.any():
            empty_fp_pixels += int(pred_counts[empty].sum(dtype=np.uint64))
            empty_pixels += int(empty.sum()) * N_PIXELS
            correct_empty_pairs += int((pred_counts[empty] == 0).sum())
            empty_pairs += int(empty.sum())
        overlap = np.count_nonzero(target.sum(axis=1) > 1)
        gt_overlap_pixels += int(overlap)
        gt_foreground_pixels += int(np.count_nonzero(target))

    foreground_dice = positive_dice_sum / max(positive_pairs, 1)
    false_positive_score = 1.0 - empty_fp_pixels / max(empty_pixels, 1)
    empty_accuracy = correct_empty_pairs / max(empty_pairs, 1)
    score = 0.70 * foreground_dice + 0.20 * false_positive_score + 0.10 * empty_accuracy
    return {
        "weighted_score": float(score),
        "foreground_dice": float(foreground_dice),
        "false_positive_score": float(false_positive_score),
        "empty_accuracy": float(empty_accuracy),
        "positive_pairs": int(positive_pairs),
        "empty_pairs": int(empty_pairs),
        "gt_pixel_overlap_pixels": int(gt_overlap_pixels),
        "gt_foreground_pixels": int(gt_foreground_pixels),
        "gt_pixel_overlap_rate": float(gt_overlap_pixels / max(gt_foreground_pixels, 1)),
    }


# Start from V19's selected per-class hard gates. Evaluate soft factors against that exact CAL-only baseline.
V19_CLASSIFIER_POLICY = {
    name: {"mode": "hard", "gate": CLASS_GATE_THRESHOLDS[name]}
    for name in CLASSES
}
V19_GATE_SEARCH_ROWS = []
for class_idx, class_name in enumerate(CLASSES):
    baseline_policy = {"mode": "hard", "gate": CLASS_GATE_THRESHOLDS[class_name]}
    baseline_contribution = v19_class_contribution(CAL_IDX, class_idx, baseline_policy)
    candidate_rows = []
    for factor in V19_SOFT_FACTORS:
        candidate_policy = {"mode": "soft_threshold", "factor": float(factor)}
        contribution = v19_class_contribution(CAL_IDX, class_idx, candidate_policy)
        gain = float(contribution - baseline_contribution)
        candidate_rows.append((gain, candidate_policy, contribution))
        V19_GATE_SEARCH_ROWS.append({
            "class": class_name,
            "policy": "soft_threshold",
            "factor": float(factor),
            "calibration_contribution": float(contribution),
            "baseline_hard_contribution": float(baseline_contribution),
            "gain": gain,
        })
    best_gain, best_policy, best_contribution = max(candidate_rows, key=lambda x: x[0])
    minimum_gain = float(V19_MIN_SOFT_GAIN[class_name])
    if best_gain >= minimum_gain:
        V19_CLASSIFIER_POLICY[class_name] = best_policy
        decision = "soft threshold accepted"
    else:
        decision = "retain V19 hard classifier gate"
    print(
        f"{class_name}: {decision} | policy={V19_CLASSIFIER_POLICY[class_name]} "
        f"| candidate gain={best_gain:+.6f} | required={minimum_gain:.6f}"
    )
    V19_GATE_SEARCH_ROWS.append({
        "class": class_name,
        "policy": "selected",
        "factor": V19_CLASSIFIER_POLICY[class_name].get("factor"),
        "calibration_contribution": float(best_contribution if best_gain >= minimum_gain else baseline_contribution),
        "baseline_hard_contribution": float(baseline_contribution),
        "gain": float(best_gain if best_gain >= minimum_gain else 0.0),
        "decision": decision,
    })

# Report V19 hard-gate baseline and V19 combined soft-gate result on CAL and reserved HOLDOUT.
V19_BASELINE_POLICY = {
    name: {"mode": "hard", "gate": CLASS_GATE_THRESHOLDS[name]}
    for name in CLASSES
}
V19_CAL_BASELINE = v19_evaluate_policy(CAL_IDX, V19_BASELINE_POLICY, resolve_conflicts=False)
V19_HOLDOUT_BASELINE = v19_evaluate_policy(HOLDOUT_IDX, V19_BASELINE_POLICY, resolve_conflicts=False)
V19_CAL_SOFT = v19_evaluate_policy(CAL_IDX, V19_CLASSIFIER_POLICY, resolve_conflicts=False)
V19_HOLDOUT_SOFT = v19_evaluate_policy(HOLDOUT_IDX, V19_CLASSIFIER_POLICY, resolve_conflicts=False)

# Test pixel-level cross-class arbitration only when CAL labels have negligible class overlap.
V19_CAL_SOFT_EXCLUSIVE = v19_evaluate_policy(CAL_IDX, V19_CLASSIFIER_POLICY, resolve_conflicts=True)
V19_GT_OVERLAP_RATE_CAL = float(V19_CAL_SOFT["gt_pixel_overlap_rate"])
V19_MIN_COMBINED_SOFT_GAIN = 0.00015
V19_SOFT_POLICY_REVERTED = bool(
    V19_CAL_SOFT["weighted_score"] - V19_CAL_BASELINE["weighted_score"] < V19_MIN_COMBINED_SOFT_GAIN
)

if V19_SOFT_POLICY_REVERTED:
    # Reject the soft classifier policy, but do not automatically reject the
    # independent pixel-conflict experiment. Re-test conflict resolution on
    # the restored hard-gate baseline, then decide it on CAL_IDX alone.
    print("Combined soft-gate gain below acceptance margin; reverting to V19 hard-gate policy.")
    V19_CLASSIFIER_POLICY = copy.deepcopy(V19_BASELINE_POLICY)
    V19_CAL_POLICY_NO_CONFLICT = V19_CAL_BASELINE
    V19_CAL_POLICY_EXCLUSIVE = v19_evaluate_policy(
        CAL_IDX, V19_BASELINE_POLICY, resolve_conflicts=True
    )
    V19_USE_EXCLUSIVE_PIXEL_CONFLICTS = bool(
        V19_GT_OVERLAP_RATE_CAL <= V19_MAX_GT_OVERLAP_RATE
        and V19_CAL_POLICY_EXCLUSIVE["weighted_score"]
            - V19_CAL_POLICY_NO_CONFLICT["weighted_score"] >= V19_MIN_CONFLICT_GAIN
    )
    V19_CAL_FINAL = (
        V19_CAL_POLICY_EXCLUSIVE if V19_USE_EXCLUSIVE_PIXEL_CONFLICTS
        else V19_CAL_POLICY_NO_CONFLICT
    )
    V19_HOLDOUT_SELECTED = v19_evaluate_policy(
        HOLDOUT_IDX, V19_BASELINE_POLICY,
        resolve_conflicts=V19_USE_EXCLUSIVE_PIXEL_CONFLICTS
    )
else:
    # Soft gating is accepted. Pixel-conflict arbitration is an independent
    # decision evaluated against the exact same selected soft policy.
    V19_USE_EXCLUSIVE_PIXEL_CONFLICTS = bool(
        V19_GT_OVERLAP_RATE_CAL <= V19_MAX_GT_OVERLAP_RATE
        and V19_CAL_SOFT_EXCLUSIVE["weighted_score"]
            - V19_CAL_SOFT["weighted_score"] >= V19_MIN_CONFLICT_GAIN
    )
    V19_CAL_POLICY_NO_CONFLICT = V19_CAL_SOFT
    V19_CAL_POLICY_EXCLUSIVE = V19_CAL_SOFT_EXCLUSIVE
    V19_CAL_FINAL = (
        V19_CAL_POLICY_EXCLUSIVE if V19_USE_EXCLUSIVE_PIXEL_CONFLICTS
        else V19_CAL_POLICY_NO_CONFLICT
    )
    V19_HOLDOUT_SELECTED = v19_evaluate_policy(
        HOLDOUT_IDX, V19_CLASSIFIER_POLICY,
        resolve_conflicts=V19_USE_EXCLUSIVE_PIXEL_CONFLICTS
    )

V19_POLICY_REPORT = {
    "notebook_version": "v19_adaptive_soft_gate_conflict_aware_hybrid",
    "best_public_score_baseline": BEST_PUBLIC_SCORE_BASELINE,
    "selected_classifier_policy_by_class": V19_CLASSIFIER_POLICY,
    "soft_gate_factor_candidates": V19_SOFT_FACTORS,
    "pixel_conflict_resolution_selected": V19_USE_EXCLUSIVE_PIXEL_CONFLICTS,
    "calibration_gt_pixel_overlap_rate": V19_GT_OVERLAP_RATE_CAL,
    "calibration_hard_gate_baseline": V19_CAL_BASELINE,
    "calibration_selected_policy": V19_CAL_FINAL,
    "calibration_selected_policy_without_conflict_resolution": V19_CAL_POLICY_NO_CONFLICT,
    "calibration_selected_policy_with_conflict_resolution": V19_CAL_POLICY_EXCLUSIVE,
    "calibration_soft_gate_candidate_without_conflicts": V19_CAL_SOFT,
    "soft_gate_policy_reverted_to_hard_gate": V19_SOFT_POLICY_REVERTED,
    "holdout_hard_gate_baseline": V19_HOLDOUT_BASELINE,
    "holdout_selected_policy": V19_HOLDOUT_SELECTED,
    "thresholds_used_for_v19_submission": V19_FINAL_THRESHOLDS,
    "selection_split": "CAL_IDX only; HOLDOUT_IDX is diagnostic",
    "component_filter": "disabled",
    "training_performed": False,
    "caveat": "The original checkpoints were selected using the original validation workflow; the internal holdout is diagnostic, not fully independent.",
}
V19_POLICY_CONFIG_PATH = OUTPUT_DIR / "v19_policy_config.json"
with open(V19_POLICY_CONFIG_PATH, "w") as f:
    json.dump(V19_POLICY_REPORT, f, indent=2)
V19_GATE_SEARCH_PATH = OUTPUT_DIR / "v19_soft_gate_search.csv"
pd.DataFrame(V19_GATE_SEARCH_ROWS).to_csv(V19_GATE_SEARCH_PATH, index=False)

# Replace the V19 hard-gate boolean with the V19 chosen policy (may use a soft classifier threshold).
COLAB_V7_BLEND_ACTIVE = bool(
    COLAB_V7_AVAILABLE and (
        any(float(CLASS_BETA[name]) > 0.0 for name in CLASSES)
        or any(V19_CLASSIFIER_POLICY[name]["mode"] == "soft_threshold" for name in CLASSES)
        or any(CLASS_GATE_THRESHOLDS[name] is not None for name in CLASSES)
    )
)
print("\nV19 policy comparison")
print("CAL hard gate baseline:", round(V19_CAL_BASELINE["weighted_score"], 6))
print("CAL selected final policy:", round(V19_CAL_FINAL["weighted_score"], 6))
print("HOLDOUT hard gate baseline:", round(V19_HOLDOUT_BASELINE["weighted_score"], 6))
print("HOLDOUT selected policy:", round(V19_HOLDOUT_SELECTED["weighted_score"], 6))
print("Ground-truth pixel overlap rate on CAL:", round(V19_GT_OVERLAP_RATE_CAL, 6))
print("Exclusive pixel conflict resolution selected:", V19_USE_EXCLUSIVE_PIXEL_CONFLICTS)
print("V19 final thresholds (CAL-only):", V19_FINAL_THRESHOLDS)
print("Soft-gate policy reverted:", V19_SOFT_POLICY_REVERTED)
print("V19 policy config:", V19_POLICY_CONFIG_PATH)


In [ ]:
# V19 independent audit: keep only adaptive policies that survive a second CAL subset.
# Pixel thresholds, beta values, blend weights and hard-gate candidates were selected
# on CAL_IDX. This cell uses CAL_AUDIT_IDX only to accept/reject the final policy family.
# HOLDOUT_IDX remains diagnostic and is never used to choose a policy.

V19_AUDIT_MIN_GAIN = {
    "defect_a": 0.00020,
    "defect_b": 0.00075,  # Rare class: require stronger evidence.
    "defect_c": 0.00020,
    "defect_d": 0.00020,
}
V19_AUDIT_MIN_POSITIVES = 10
V19_AUDIT_CLASS_RESULTS = {}

for class_idx, class_name in enumerate(CLASSES):
    baseline_policy = V19_BASELINE_POLICY[class_name]
    selected_policy = V19_CLASSIFIER_POLICY[class_name]
    audit_positive_count = int((gt_counts[CAL_AUDIT_IDX, class_idx] > 0).sum())
    baseline_contribution = v19_class_contribution(CAL_AUDIT_IDX, class_idx, baseline_policy)

    if selected_policy == baseline_policy:
        selected_contribution = baseline_contribution
        decision = "baseline retained; no soft-policy change proposed"
        gain = 0.0
        V19_CLASSIFIER_POLICY[class_name] = copy.deepcopy(baseline_policy)
    elif audit_positive_count < V19_AUDIT_MIN_POSITIVES:
        selected_contribution = v19_class_contribution(CAL_AUDIT_IDX, class_idx, selected_policy)
        decision = "baseline retained; audit positive support too small"
        gain = float(selected_contribution - baseline_contribution)
        V19_CLASSIFIER_POLICY[class_name] = copy.deepcopy(baseline_policy)
        selected_contribution = baseline_contribution
    else:
        selected_contribution = v19_class_contribution(CAL_AUDIT_IDX, class_idx, selected_policy)
        gain = float(selected_contribution - baseline_contribution)
        required_gain = float(V19_AUDIT_MIN_GAIN[class_name])
        if gain >= required_gain:
            decision = "soft policy passed independent audit"
        else:
            decision = "baseline retained; soft policy failed independent audit"
            V19_CLASSIFIER_POLICY[class_name] = copy.deepcopy(baseline_policy)
            selected_contribution = baseline_contribution

    V19_AUDIT_CLASS_RESULTS[class_name] = {
        "audit_positive_pairs": audit_positive_count,
        "baseline_policy": baseline_policy,
        "proposed_policy": selected_policy,
        "final_policy": V19_CLASSIFIER_POLICY[class_name],
        "baseline_audit_contribution": float(baseline_contribution),
        "selected_audit_contribution": float(selected_contribution),
        "proposed_gain": float(gain),
        "minimum_gain": float(V19_AUDIT_MIN_GAIN[class_name]),
        "decision": decision,
    }
    print(
        f"{class_name}: {decision} | audit positives={audit_positive_count} "
        f"| proposed gain={gain:+.6f}"
    )

# Pixel-conflict arbitration is a joint four-class policy. Audit it globally after
# the per-class soft policies have been accepted/rejected.
V19_AUDIT_NO_CONFLICT = v19_evaluate_policy(
    CAL_AUDIT_IDX, V19_CLASSIFIER_POLICY, resolve_conflicts=False
)
V19_AUDIT_WITH_CONFLICT = v19_evaluate_policy(
    CAL_AUDIT_IDX, V19_CLASSIFIER_POLICY, resolve_conflicts=True
)
V19_AUDIT_CONFLICT_GAIN = float(
    V19_AUDIT_WITH_CONFLICT["weighted_score"] - V19_AUDIT_NO_CONFLICT["weighted_score"]
)
V19_AUDIT_GT_OVERLAP_RATE = float(V19_AUDIT_NO_CONFLICT["gt_pixel_overlap_rate"])
V19_USE_EXCLUSIVE_PIXEL_CONFLICTS = bool(
    V19_AUDIT_GT_OVERLAP_RATE <= V19_MAX_GT_OVERLAP_RATE
    and V19_AUDIT_CONFLICT_GAIN >= V19_MIN_CONFLICT_GAIN
)

V19_AUDIT_SELECTED = (
    V19_AUDIT_WITH_CONFLICT if V19_USE_EXCLUSIVE_PIXEL_CONFLICTS
    else V19_AUDIT_NO_CONFLICT
)
V19_CAL_FINAL = v19_evaluate_policy(
    CAL_IDX, V19_CLASSIFIER_POLICY,
    resolve_conflicts=V19_USE_EXCLUSIVE_PIXEL_CONFLICTS
)
V19_HOLDOUT_SELECTED = v19_evaluate_policy(
    HOLDOUT_IDX, V19_CLASSIFIER_POLICY,
    resolve_conflicts=V19_USE_EXCLUSIVE_PIXEL_CONFLICTS
)

# Recompute whether the auxiliary branch contributes to final test inference.
COLAB_V7_BLEND_ACTIVE = bool(
    COLAB_V7_AVAILABLE and (
        any(float(CLASS_BETA[name]) > 0.0 for name in CLASSES)
        or any(V19_CLASSIFIER_POLICY[name]["mode"] == "soft_threshold" for name in CLASSES)
        or any(V19_CLASSIFIER_POLICY[name].get("gate") is not None for name in CLASSES)
    )
)

V19_AUDIT_REPORT = {
    "notebook_version": "v19_audit_gated_eight_model_hybrid",
    "cal_tune_rows": int(len(CAL_IDX)),
    "cal_audit_rows": int(len(CAL_AUDIT_IDX)),
    "holdout_rows": int(len(HOLDOUT_IDX)),
    "audit_split_note": CAL_AUDIT_SPLIT_NOTE,
    "class_policy_audit": V19_AUDIT_CLASS_RESULTS,
    "audit_no_conflict": V19_AUDIT_NO_CONFLICT,
    "audit_with_conflict": V19_AUDIT_WITH_CONFLICT,
    "audit_conflict_gain": V19_AUDIT_CONFLICT_GAIN,
    "audit_gt_pixel_overlap_rate": V19_AUDIT_GT_OVERLAP_RATE,
    "exclusive_pixel_conflicts_selected": bool(V19_USE_EXCLUSIVE_PIXEL_CONFLICTS),
    "audit_selected_score": float(V19_AUDIT_SELECTED["weighted_score"]),
    "holdout_selected_score_diagnostic_only": float(V19_HOLDOUT_SELECTED["weighted_score"]),
    "holdout_used_for_selection": False,
}
V19_POLICY_REPORT.update(V19_AUDIT_REPORT)
V19_POLICY_CONFIG_PATH = OUTPUT_DIR / "v19_policy_config.json"
with open(V19_POLICY_CONFIG_PATH, "w") as f:
    json.dump(V19_POLICY_REPORT, f, indent=2)

print("\nV19 independent policy audit")
print("CAL_AUDIT no-conflict:", round(V19_AUDIT_NO_CONFLICT["weighted_score"], 6))
print("CAL_AUDIT with-conflict:", round(V19_AUDIT_WITH_CONFLICT["weighted_score"], 6))
print("CAL_AUDIT conflict gain:", round(V19_AUDIT_CONFLICT_GAIN, 6))
print("Pixel conflict resolution retained:", V19_USE_EXCLUSIVE_PIXEL_CONFLICTS)
print("Final V19 holdout diagnostic (not used for selection):", round(V19_HOLDOUT_SELECTED["weighted_score"], 6))
print("Final classifier policies:", V19_CLASSIFIER_POLICY)
print("Final V19 policy config:", V19_POLICY_CONFIG_PATH)


In [ ]:
SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"
sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH,
    dtype={"ID": str, "image_id": str, "defect_type": str, "mask_rle": str},
    keep_default_na=False
)

EXPECTED_COLUMNS = ["ID", "image_id", "defect_type", "mask_rle"]
if sample_submission.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected sample submission columns: {sample_submission.columns.tolist()}")

if sample_submission.duplicated(subset=["image_id", "defect_type"]).any():
    raise ValueError("Sample submission contains duplicate image-class pairs.")

if set(sample_submission["defect_type"].unique()) != set(CLASSES):
    raise ValueError("Sample submission class names do not match the model class order.")

TEST_IDS = sample_submission["image_id"].drop_duplicates().astype(str).tolist()
if len(TEST_IDS) != 3000 or len(sample_submission) != 12000:
    raise ValueError(
        f"Unexpected test template dimensions: {len(TEST_IDS)} images, {len(sample_submission)} rows."
    )

class SurfaceGuardTestDataset(Dataset):
    def __init__(self, image_ids, data_root, resample):
        self.image_ids = list(map(str, image_ids))
        self.data_root = Path(data_root)
        self.resample = resample

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        image_path = self.data_root / "test" / f"{image_id}.jpg"
        if not image_path.is_file():
            raise FileNotFoundError(f"Test image not found: {image_path}")
        with Image.open(image_path) as source:
            image = source.convert("L").resize((WIDTH, HEIGHT), resample=self.resample)
            image = np.asarray(image, dtype=np.float32) / 255.0
        mean = float(image.mean())
        std = max(float(image.std()), 0.05)
        image = np.ascontiguousarray((image - mean) / std, dtype=np.float32)
        return torch.from_numpy(image).unsqueeze(0), image_id


test_dataset_main = SurfaceGuardTestDataset(TEST_IDS, DATA_ROOT, MAIN_RESAMPLE)
test_dataset_mit = SurfaceGuardTestDataset(TEST_IDS, DATA_ROOT, MIT_RESAMPLE)

test_loader_main = DataLoader(
    test_dataset_main,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False
)
test_loader_mit = DataLoader(
    test_dataset_mit,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False
)

if len(test_loader_main) != len(test_loader_mit):
    raise RuntimeError("Main and MiT test loaders have different batch counts.")

print("Test images:", len(TEST_IDS))
print("Submission rows:", len(sample_submission))
print("Test batches:", len(test_loader_main))
print("Batch size:", TEST_BATCH_SIZE)
print("Official test template check: PASSED")

In [ ]:
def rle_encode_column_major(mask):
    pixels = np.asarray(mask, dtype=np.uint8).reshape(-1, order="F")
    if not np.any(pixels):
        return " "
    padded = np.concatenate([
        np.array([0], dtype=np.uint8),
        pixels,
        np.array([0], dtype=np.uint8)
    ])
    changes = np.flatnonzero(padded[1:] != padded[:-1]) + 1
    starts = changes[0::2]
    ends = changes[1::2]
    lengths = ends - starts
    encoded = np.empty(starts.size * 2, dtype=np.int64)
    encoded[0::2] = starts
    encoded[1::2] = lengths
    return " ".join(map(str, encoded.tolist()))


def rle_decode_for_check(rle, height=HEIGHT, width=WIDTH):
    flat = np.zeros(height * width, dtype=np.uint8)
    if str(rle).strip() == "":
        return flat.reshape((height, width), order="F")
    values = np.asarray(str(rle).split(), dtype=np.int64)
    if len(values) % 2:
        raise ValueError("RLE does not contain start-length pairs.")
    starts = values[0::2] - 1
    lengths = values[1::2]
    for start, length in zip(starts, lengths):
        if start < 0 or length <= 0 or start + length > height * width:
            raise ValueError("RLE run is outside image bounds.")
        flat[start:start + length] = 1
    return flat.reshape((height, width), order="F")


def validate_rle(rle, height=HEIGHT, width=WIDTH):
    if not isinstance(rle, str):
        return False
    if rle.strip() == "":
        return True
    try:
        values = np.asarray(rle.split(), dtype=np.int64)
        if len(values) % 2:
            return False
        starts = values[0::2]
        lengths = values[1::2]
        if np.any(starts < 1) or np.any(lengths < 1):
            return False
        ends = starts + lengths - 1
        if np.any(ends > height * width):
            return False
        if len(starts) > 1 and np.any(starts[1:] <= ends[:-1]):
            return False
        return True
    except (ValueError, TypeError, OverflowError):
        return False


# Explicit RLE unit test: the second run starts at position 9 in column-major order.
_rle_test = np.zeros((3, 4), dtype=np.uint8)
_rle_test[0, 0] = 1
_rle_test[1, 0] = 1
_rle_test[2, 2] = 1
_rle_text = rle_encode_column_major(_rle_test)
assert _rle_text == "1 2 9 1", f"RLE self-test failed: {_rle_text}"
assert np.array_equal(rle_decode_for_check(_rle_text, 3, 4), _rle_test)
assert validate_rle(_rle_text, 3, 4)
print("Column-major RLE encode/decode self-test: PASSED")

In [ ]:
print("FINAL TEST INFERENCE — V19 ADAPTIVE-GATE EIGHT-MODEL HYBRID")
print("=" * 70)
print("Optional Colab V7 checkpoints available:", COLAB_V7_AVAILABLE)
print("Colab V7 contribution/gate selected:", COLAB_V7_BLEND_ACTIVE)
print("Class-aware V13 weights:", FINAL_BLEND_CONFIG["baseline_class_aware_weights"])
print("Colab V7 beta by class:", CLASS_BETA)
print("Colab V7 classifier gates:", CLASS_GATE_THRESHOLDS)
print("Colab V7 fusion modes:", CLASS_FUSION_MODE)
print("Main inference mode:", MAIN_INFERENCE_MODE)
print("MiT inference mode:", MIT_INFERENCE_MODE)
print("V19 CAL-only pixel thresholds:", V19_FINAL_THRESHOLDS)
print("V19 classifier policy:", V19_CLASSIFIER_POLICY)
print("V19 pixel conflict resolution:", V19_USE_EXCLUSIVE_PIXEL_CONFLICTS)

if not np.allclose(CLASS_WEIGHT_MATRIX.sum(axis=1), 1.0, atol=1e-6):
    raise RuntimeError("One or more class-specific V13 weight rows do not sum to one.")

prediction_lookup = {}
processed_images = 0
start_time = time.time()
progress_path = OUTPUT_DIR / "test_predictions_progress.csv"


def save_test_progress(predictions, processed_count):
    rows = [
        {"image_id": image_id, "defect_type": class_name,
         "mask_rle": predictions[(image_id, class_name)]}
        for image_id in TEST_IDS[:processed_count]
        for class_name in CLASSES
        if (image_id, class_name) in predictions
    ]
    progress_df = pd.DataFrame(rows, columns=["image_id", "defect_type", "mask_rle"])
    temp_path = progress_path.with_name(progress_path.stem + ".tmp.csv")
    progress_df.to_csv(temp_path, index=False)
    os.replace(temp_path, progress_path)


with torch.inference_mode():
    loader_iterator = zip(test_loader_main, test_loader_mit)
    for (main_images, main_ids), (mit_images, mit_ids) in tqdm(
        loader_iterator, total=len(test_loader_main), desc="Test inference: V19 eight-model adaptive hybrid"
    ):
        main_ids = [str(x) for x in main_ids]
        mit_ids = [str(x) for x in mit_ids]
        if main_ids != mit_ids:
            raise RuntimeError("Main and MiT test-loader image order mismatch.")

        # Run the optional auxiliary group first. If it triggers CUDA OOM, the
        # global device is switched before the V13 models are evaluated.
        if COLAB_V7_BLEND_ACTIVE:
            colab_seg, colab_cls = predict_colab_v7_group(main_images)
        else:
            colab_seg, colab_cls = None, None

        p4_probs, p5_probs, p6_probs = predict_main_models(main_images)
        mit_probs = predict_mit_model(mit_images)

        # A later model can still trigger CUDA OOM after an earlier model has
        # returned CUDA tensors. Re-home every probability tensor to the current
        # DEVICE before any arithmetic to avoid CPU/CUDA device-mismatch errors.
        p4_probs = p4_probs.to(DEVICE).half().float()
        p5_probs = p5_probs.to(DEVICE).half().float()
        p6_probs = p6_probs.to(DEVICE).half().float()
        mit_probs = mit_probs.to(DEVICE).half().float()

        # Recreate small tensors inside the loop because DEVICE can change after
        # automatic OOM recovery; tensors created before the fallback become stale.
        class_weight_tensors = torch.as_tensor(
            CLASS_WEIGHT_MATRIX, dtype=torch.float32, device=DEVICE
        )
        baseline_probs = (
            class_weight_tensors[:, 0].view(1, N_CLASSES, 1, 1) * p4_probs
            + class_weight_tensors[:, 1].view(1, N_CLASSES, 1, 1) * p5_probs
            + class_weight_tensors[:, 2].view(1, N_CLASSES, 1, 1) * p6_probs
            + class_weight_tensors[:, 3].view(1, N_CLASSES, 1, 1) * mit_probs
        )
        threshold_tensor = torch.as_tensor(
            [V19_FINAL_THRESHOLDS[name] for name in CLASSES], dtype=torch.float32, device=DEVICE
        ).view(1, N_CLASSES, 1, 1)

        if COLAB_V7_BLEND_ACTIVE:
            colab_seg = colab_seg.to(DEVICE).half().float()
            colab_cls = colab_cls.to(DEVICE).half().float()
            beta_tensor = torch.as_tensor(
                [CLASS_BETA[name] for name in CLASSES], dtype=torch.float32, device=DEVICE
            ).view(1, N_CLASSES, 1, 1)
            # V19 policies: hard classifier gate or soft confidence-scaled pixel threshold.
            gate_pass = torch.ones((len(main_ids), N_CLASSES), dtype=torch.bool, device=DEVICE)
            effective_threshold_tensor = threshold_tensor.expand(len(main_ids), -1, -1, -1).clone()
            for class_index, class_name in enumerate(CLASSES):
                policy = V19_CLASSIFIER_POLICY[class_name]
                if policy["mode"] == "hard":
                    hard_gate = policy.get("gate")
                    if hard_gate is not None:
                        gate_pass[:, class_index] = colab_cls[:, class_index] >= float(hard_gate)
                elif policy["mode"] == "soft_threshold":
                    factor = float(policy["factor"])
                    q = colab_cls[:, class_index]
                    effective_threshold_tensor[:, class_index, 0, 0] = (
                        threshold_tensor[:, class_index, 0, 0]
                        * (factor - (factor - 1.0) * q)
                    )
                else:
                    raise ValueError(f"Unknown V19 classifier policy for {class_name}: {policy}")
            probability_combined = (1.0 - beta_tensor) * baseline_probs + beta_tensor * colab_seg
            eps = 1e-4
            base_clipped = baseline_probs.clamp(min=eps, max=1.0 - eps)
            colab_clipped = colab_seg.clamp(min=eps, max=1.0 - eps)
            logit_combined = torch.sigmoid(
                (1.0 - beta_tensor) * torch.logit(base_clipped)
                + beta_tensor * torch.logit(colab_clipped)
            )
            logit_mode_tensor = torch.as_tensor(
                [CLASS_FUSION_MODE[name] == "logit" for name in CLASSES],
                dtype=torch.bool, device=DEVICE
            ).view(1, N_CLASSES, 1, 1)
            combined_probs = torch.where(logit_mode_tensor, logit_combined, probability_combined)
            del colab_seg, colab_cls, beta_tensor
            del probability_combined, base_clipped, colab_clipped, logit_combined, logit_mode_tensor
        else:
            combined_probs = baseline_probs
            gate_pass = torch.ones(
                (len(main_ids), N_CLASSES), dtype=torch.bool, device=DEVICE
            )
            effective_threshold_tensor = threshold_tensor.expand(len(main_ids), -1, -1, -1)

        if not torch.isfinite(combined_probs).all():
            raise RuntimeError("Non-finite values in final blended test probabilities.")
        binary_mask_bool = (combined_probs >= effective_threshold_tensor) & gate_pass[:, :, None, None]
        if V19_USE_EXCLUSIVE_PIXEL_CONFLICTS:
            # Among already-positive class masks, keep the class with the largest
            # probability / effective-threshold margin. Empty pixels remain empty.
            normalized_margin = combined_probs / effective_threshold_tensor.clamp(min=1e-6)
            normalized_margin = torch.where(
                binary_mask_bool, normalized_margin, torch.full_like(normalized_margin, -torch.inf)
            )
            winners = torch.argmax(normalized_margin, dim=1)
            class_ids = torch.arange(N_CLASSES, device=DEVICE).view(1, N_CLASSES, 1, 1)
            binary_mask_bool = binary_mask_bool & (winners[:, None, :, :] == class_ids)
        binary_masks = binary_mask_bool.to(torch.uint8).cpu().numpy()
        for batch_index in range(len(main_ids)):
            for class_index, class_name in enumerate(CLASSES):
                min_area = int(COMPONENT_MIN_AREAS.get(class_name, 0))
                if min_area > 1:
                    binary_masks[batch_index, class_index] = remove_small_components(
                        binary_masks[batch_index, class_index], min_area
                    ).astype(np.uint8)

        for batch_index, image_id in enumerate(main_ids):
            for class_index, class_name in enumerate(CLASSES):
                prediction_lookup[(image_id, class_name)] = rle_encode_column_major(
                    binary_masks[batch_index, class_index]
                )

        processed_images += len(main_ids)
        if processed_images % 100 == 0 or processed_images == len(TEST_IDS):
            save_test_progress(prediction_lookup, processed_images)

        del p4_probs, p5_probs, p6_probs, mit_probs, baseline_probs, combined_probs, binary_masks, gate_pass
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

if processed_images != len(TEST_IDS):
    raise RuntimeError(f"Test inference incomplete: {processed_images}/{len(TEST_IDS)} images.")
expected_pairs = len(TEST_IDS) * N_CLASSES
if len(prediction_lookup) != expected_pairs:
    raise RuntimeError(f"Prediction coverage mismatch: {len(prediction_lookup)} != {expected_pairs}.")

submission = sample_submission.copy()
submission["mask_rle"] = [
    prediction_lookup[(str(image_id), str(class_name))]
    for image_id, class_name in zip(submission["image_id"], submission["defect_type"])
]
submission["mask_rle"] = submission["mask_rle"].replace("", " ")

FINAL_SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
temporary_submission_path = FINAL_SUBMISSION_PATH.with_name("submission.tmp.csv")
submission.to_csv(temporary_submission_path, index=False)
os.replace(temporary_submission_path, FINAL_SUBMISSION_PATH)
backup_path = OUTPUT_DIR / "submission_surfaceguard_v19_adaptive_gate_conflict_aware.csv"
submission.to_csv(backup_path, index=False)

elapsed_minutes = (time.time() - start_time) / 60.0
print("\nTest inference completed.")
print("Images processed:", processed_images)
print("Submission rows:", len(submission))
print("Elapsed minutes:", round(elapsed_minutes, 2))
print("Saved candidate:", FINAL_SUBMISSION_PATH)
print("Progress file:", progress_path)

check = pd.read_csv(
    FINAL_SUBMISSION_PATH,
    dtype={"ID": str, "image_id": str, "defect_type": str, "mask_rle": str},
    keep_default_na=False,
)
if check.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f"Wrong submission columns: {check.columns.tolist()}")
if check.shape != (12000, 4):
    raise ValueError(f"Wrong submission shape: {check.shape}")
for column in ["ID", "image_id", "defect_type"]:
    if check[column].tolist() != sample_submission[column].astype(str).tolist():
        raise ValueError(f"Submission template order mismatch in {column}.")
if check.isna().any().any():
    raise ValueError("Null values found in saved submission.")
if check.duplicated().any() or check.duplicated(subset=["image_id", "defect_type"]).any():
    raise ValueError("Duplicate rows or image-class pairs found in saved submission.")
if check["ID"].nunique() != 12000 or check["image_id"].nunique() != 3000:
    raise ValueError("Submission ID/image counts are incorrect.")
if not (check.groupby("image_id").size() == N_CLASSES).all():
    raise ValueError("Not every test image has exactly four class rows.")
class_counts = check["defect_type"].value_counts().reindex(CLASSES, fill_value=0)
if not (class_counts == 3000).all():
    raise ValueError(f"Unexpected class counts:\n{class_counts}")
rle_valid = check["mask_rle"].map(validate_rle)
if not rle_valid.all():
    display(check.loc[~rle_valid].head(10))
    raise ValueError(f"Invalid RLE rows: {int((~rle_valid).sum())}")

empty_masks = int(check["mask_rle"].str.strip().eq("").sum())
nonempty_masks = int(check["mask_rle"].str.strip().ne("").sum())
file_size_mb = FINAL_SUBMISSION_PATH.stat().st_size / (1024 ** 2)
for row_index in [0, 1, 2, 100, 1000, 5000, 11999]:
    image_id = str(check.iloc[row_index]["image_id"])
    class_name = str(check.iloc[row_index]["defect_type"])
    rle = prediction_lookup[(image_id, class_name)]
    decoded = rle_decode_for_check(rle)
    if not validate_rle(rle) or decoded.shape != (HEIGHT, WIDTH):
        raise RuntimeError(f"RLE round-trip check failed at row {row_index}.")

FINAL_REPORT = {
    "status": "PASSED",
    "notebook_version": "v19_adaptive_soft_gate_conflict_aware_hybrid",
    "best_previous_public_score": BEST_PUBLIC_SCORE_BASELINE,
    "v19_classifier_policy": V19_CLASSIFIER_POLICY,
    "v19_pixel_conflict_resolution": bool(V19_USE_EXCLUSIVE_PIXEL_CONFLICTS),
    "v19_policy_holdout_metrics": V19_HOLDOUT_SELECTED,
    "v19_calibration_policy_metrics": V19_CAL_FINAL,
    "v19_soft_gate_candidate_calibration_metrics": V19_CAL_SOFT,
    "v19_soft_gate_policy_reverted": V19_SOFT_POLICY_REVERTED,
    "v19_policy_config_path": str(V19_POLICY_CONFIG_PATH),
    "public_score_goal": 0.68,
    "colab_v7_checkpoints_available": bool(COLAB_V7_AVAILABLE),
    "colab_v7_auxiliary_models_used_in_test": bool(COLAB_V7_BLEND_ACTIVE),
    "colab_v7_missing_checkpoints": COLAB_V7_MISSING,
    "device": str(DEVICE),
    "gpu_name": GPU_NAME,
    "main_inference_mode": MAIN_INFERENCE_MODE,
    "mit_inference_mode": MIT_INFERENCE_MODE,
    "beta_by_class": CLASS_BETA,
    "fusion_mode_by_class": CLASS_FUSION_MODE,
    "component_min_area_by_class": COMPONENT_MIN_AREAS,
    "gate_thresholds": CLASS_GATE_THRESHOLDS,
    "pixel_thresholds": V19_FINAL_THRESHOLDS,
    "holdout_hybrid_score": float(hybrid_holdout_metrics["weighted_score"]),
    "holdout_hybrid_score_without_component_filter": float(hybrid_holdout_metrics_unfiltered["weighted_score"]),
    "holdout_v13_baseline_score": float(v13_baseline_holdout_metrics["weighted_score"]),
    "full_validation_calibrated_objective_without_component_filter": float(full_validation_score),
    "full_validation_score_with_component_filter": (
        float(full_validation_metrics_with_components["weighted_score"])
        if full_validation_metrics_with_components is not None else None
    ),
    "calibration_score": float(hybrid_calibration_score),
    "metric_contribution_sum_assertion": "passed" if COLAB_V7_AVAILABLE else "not_applicable_fallback",
    "submission_file": str(FINAL_SUBMISSION_PATH),
    "rows": int(len(check)),
    "unique_images": int(check["image_id"].nunique()),
    "class_counts": {str(k): int(v) for k, v in class_counts.items()},
    "empty_masks": empty_masks,
    "nonempty_masks": nonempty_masks,
    "invalid_rle_rows": int((~rle_valid).sum()),
    "null_values": int(check.isna().sum().sum()),
    "duplicate_rows": int(check.duplicated().sum()),
    "file_size_mb": round(file_size_mb, 3),
    "metric": "0.70 foreground Dice + 0.20 false-positive score + 0.10 empty-prediction accuracy",
    "metric_uses_global_pair_denominators": True,
    "training_performed": False,
    "holdout_caveat": "The checkpoints were selected/trained using prior validation labels; holdout is diagnostic, not an untouched independent estimate.",
    "public_score_caveat": "The Kaggle public score is only known after submission; local validation does not guarantee improvement over 0.64701.",
}
FINAL_REPORT_PATH = OUTPUT_DIR / "final_submission_validation_report.json"
with open(FINAL_REPORT_PATH, "w") as f:
    json.dump(FINAL_REPORT, f, indent=2)
FINAL_CLASS_ANALYSIS_PATH = OUTPUT_DIR / "final_validation_per_class_metrics.csv"
pd.DataFrame(hybrid_holdout_metrics["class_metrics"]).T.to_csv(FINAL_CLASS_ANALYSIS_PATH)

print("=" * 70)
print("FINAL SUBMISSION VALIDATION: PASSED")
print("=" * 70)
print("Submission:", FINAL_SUBMISSION_PATH)
print("Rows:", len(check), "| Unique images:", check["image_id"].nunique())
print("Class counts:\n", class_counts)
print("Empty masks:", empty_masks, "| Non-empty masks:", nonempty_masks)
print("Invalid RLE rows:", int((~rle_valid).sum()))
print("Device:", DEVICE, "| GPU:", GPU_NAME)
print("Colab V7 checkpoints available:", COLAB_V7_AVAILABLE)
print("Colab V7 contribution used in test:", COLAB_V7_BLEND_ACTIVE)
print("Hybrid holdout:", round(hybrid_holdout_metrics["weighted_score"], 6))
print("V13 baseline holdout:", round(v13_baseline_holdout_metrics["weighted_score"], 6))
print("Full-validation objective:", round(full_validation_score, 6))
print("Validation report:", FINAL_REPORT_PATH)


In [ ]:
# This optional run records the V19 eight-model policy evaluation. Missing secrets or
# W&B connectivity do not affect submission generation.
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    if api_key and api_key.strip():
        WANDB_HOST = "https://api.forge.coreweave.com"
        WANDB_ENTITY = "23f3003529-dl-genai-project"
        WANDB_PROJECT = "23f3003529-t32026"

        wandb.login(
            key=api_key.strip(),
            host=WANDB_HOST,
            relogin=True
        )
        del api_key

        final_run = wandb.init(
            entity=WANDB_ENTITY,
            project=WANDB_PROJECT,
            name="SurfaceGuard-Version19-Adaptive-Gate-Conflict-Aware-Hybrid",
            job_type="ensemble-evaluation",
            tags=["SurfaceGuardAI", "eight-model-hybrid", "class-aware-weights", "inference-only"],
            config={
                "models": FINAL_BLEND_CONFIG["models"] + (["ColabV7 A/B/C/D"] if COLAB_V7_AVAILABLE else []),
                "class_aware_weights": FINAL_BLEND_CONFIG["baseline_class_aware_weights"],
                "main_inference_mode": MAIN_INFERENCE_MODE,
                "main_tta_passes": len(MAIN_TTA_TRANSFORMS),
                "mit_b4_inference_mode": MIT_INFERENCE_MODE,
                "version": "v19_adaptive_soft_gate_conflict_aware_hybrid",
                "main_resampling": "bilinear",
                "weight_optimisation": "class_aware_v4_v5_v6_mit_b4_with_fine_0.05_pairwise_refinement_plus_expanded_colab_v7_beta_gate_and_local_logit_fusion",
                "colab_v7_available": bool(COLAB_V7_AVAILABLE),
                "colab_v7_beta_by_class": CLASS_BETA,
                "colab_v7_gate_thresholds": CLASS_GATE_THRESHOLDS,
                "colab_v7_fusion_mode_by_class": CLASS_FUSION_MODE,
                "component_min_area_by_class": COMPONENT_MIN_AREAS,
                "component_min_area_candidate_by_class": COMPONENT_MIN_AREAS_CALIBRATED,
                "mit_resampling": "bilinear",
                "execution_device": str(DEVICE),
                "gpu_name": GPU_NAME,
                "previous_public_baseline": BEST_PUBLIC_SCORE_BASELINE,
                "v19_classifier_policy": V19_CLASSIFIER_POLICY,
                "v19_pixel_conflict_resolution": V19_USE_EXCLUSIVE_PIXEL_CONFLICTS,
                "thresholds": V19_FINAL_THRESHOLDS,
                "target_local_validation_score": LOCAL_VALIDATION_TARGET,
                "public_leaderboard_goal": PUBLIC_SCORE_TARGET
            }
        )
        final_run.summary.update({
            "component_candidate_holdout_score": float(hybrid_holdout_metrics_component_candidate["weighted_score"]),
            "calibration_score": float(hybrid_calibration_score),
            "holdout_score": float(hybrid_holdout_metrics["weighted_score"]),
            "v19_holdout_score": float(V19_HOLDOUT_SELECTED["weighted_score"]),
            "v19_calibration_score": float(V19_CAL_FINAL["weighted_score"]),
            "holdout_score_without_component_filter": float(hybrid_holdout_metrics_unfiltered["weighted_score"]),
            "calibration_score": float(hybrid_calibration_score),
            "full_validation_calibrated_objective": float(full_validation_score),
            "submission_rows": int(len(check)),
            "valid_rle_rows": int(rle_valid.sum()),
            "empty_masks": empty_masks,
            "submission_validation": "passed"
        })
        final_run.finish()
        print("W&B final evaluation run:", final_run.url)
    else:
        print("W&B skipped: WANDB_API_KEY secret is empty.")
except Exception as exc:
    print("W&B logging skipped; submission generation is unaffected.")
    print(type(exc).__name__ + ":", str(exc)[:300])


In [ ]:
import os
import shutil
from pathlib import Path
from IPython.display import FileLink, display

print("SURFACEGUARD AI — VERSION 19 FINAL ARTIFACTS")
print("=" * 60)

# The inference cell creates this validated candidate inside OUTPUT_DIR.
CANDIDATE_SUBMISSION_PATH = OUTPUT_DIR / "submission_surfaceguard_v19_adaptive_gate_conflict_aware.csv"
KAGGLE_SUBMISSION_PATH = Path("/kaggle/working/submission.csv")

if not CANDIDATE_SUBMISSION_PATH.is_file():
    raise FileNotFoundError(
        "The final candidate submission was not created by the inference cell: "
        f"{CANDIDATE_SUBMISSION_PATH}"
    )

# Copy atomically so Kaggle always sees the exact required filename at the root.
TEMP_SUBMISSION_COPY = KAGGLE_SUBMISSION_PATH.with_name("submission.copy.tmp.csv")
shutil.copy2(CANDIDATE_SUBMISSION_PATH, TEMP_SUBMISSION_COPY)
os.replace(TEMP_SUBMISSION_COPY, KAGGLE_SUBMISSION_PATH)

# Verify the copied root-level file, not just the source candidate.
root_check = pd.read_csv(
    KAGGLE_SUBMISSION_PATH,
    dtype={"ID": str, "image_id": str, "defect_type": str, "mask_rle": str},
    keep_default_na=False
)
candidate_check = pd.read_csv(
    CANDIDATE_SUBMISSION_PATH,
    dtype={"ID": str, "image_id": str, "defect_type": str, "mask_rle": str},
    keep_default_na=False
)

if not root_check.equals(candidate_check):
    raise ValueError("Root submission.csv differs from the validated candidate submission.")

if root_check.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f"Copied submission has incorrect columns: {root_check.columns.tolist()}")
if root_check.shape != (12000, 4):
    raise ValueError(f"Copied submission has incorrect shape: {root_check.shape}")
if root_check.isna().any().any():
    raise ValueError("Copied submission contains null values.")
if root_check.duplicated(subset=["image_id", "defect_type"]).any():
    raise ValueError("Copied submission contains duplicate image-class pairs.")
if root_check["ID"].tolist() != sample_submission["ID"].astype(str).tolist():
    raise ValueError("Copied submission ID order differs from the sample template.")
if root_check["image_id"].tolist() != sample_submission["image_id"].astype(str).tolist():
    raise ValueError("Copied submission image order differs from the sample template.")
if root_check["defect_type"].tolist() != sample_submission["defect_type"].astype(str).tolist():
    raise ValueError("Copied submission class order differs from the sample template.")
if not root_check["mask_rle"].map(validate_rle).all():
    raise ValueError("Copied submission contains invalid RLE masks.")

print("Kaggle submission file copied and verified:", KAGGLE_SUBMISSION_PATH)
print("Rows:", len(root_check))
print("File size (MB):", round(KAGGLE_SUBMISSION_PATH.stat().st_size / (1024 ** 2), 3))

final_artifacts = [
    KAGGLE_SUBMISSION_PATH,
    CANDIDATE_SUBMISSION_PATH,
    FINAL_REPORT_PATH,
    FINAL_CLASS_ANALYSIS_PATH,
    CONFIG_PATH,
    OUTPUT_DIR / "global_four_model_weight_search.csv",
    OUTPUT_DIR / "fine_four_model_weight_search.csv",
    V19_POLICY_CONFIG_PATH,
    V19_GATE_SEARCH_PATH,
    OUTPUT_DIR / "class_aware_weight_selection.csv",
    OUTPUT_DIR / "hybrid_beta_gate_search.csv",
    OUTPUT_DIR / "hybrid_class_selection.csv",
    OUTPUT_DIR / "component_min_area_search.csv",
    OUTPUT_DIR / "test_predictions_progress.csv"
]

print("\nAVAILABLE OUTPUT ARTIFACTS")
for artifact in final_artifacts:
    artifact = Path(artifact)
    if not artifact.is_file():
        raise FileNotFoundError(f"Expected output is missing: {artifact}")
    print("\n", artifact.name, "|", round(artifact.stat().st_size / (1024 ** 2), 3), "MB")
    display(FileLink(str(artifact)))

print("\nFINAL NOTEBOOK PIPELINE COMPLETED.")
print("Colab V7 auxiliary models included:", COLAB_V7_AVAILABLE)
print("V19 adaptive classifier policy:", V19_CLASSIFIER_POLICY)
print("V19 pixel conflict resolution:", V19_USE_EXCLUSIVE_PIXEL_CONFLICTS)
print("If False, this was the V13 fallback rather than the new hybrid experiment.")
print("No model training was performed. The notebook ran checkpoint loading, validation/test inference, CAL-only policy selection, holdout diagnostics and submission validation.")


In [ ]:
import pandas as pd
from pathlib import Path

files_to_repair = [
    Path("/kaggle/working/submission.csv"),
    OUTPUT_DIR / "submission_surfaceguard_v19_adaptive_gate_conflict_aware.csv"
]

for path in files_to_repair:

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False
    )

    # Same correction used in your previously accepted notebook
    original_empty_count = int(
        df["mask_rle"].eq("").sum()
    )

    df["mask_rle"] = df["mask_rle"].replace("", " ")

    df.to_csv(path, index=False)

    # Verify the saved CSV
    check = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False
    )

    assert len(check) == 12000
    assert check["mask_rle"].eq("").sum() == 0
    assert check.isna().sum().sum() == 0
    assert check["mask_rle"].map(validate_rle).all()

    print("\nREPAIR PASSED:", path.name)
    print("Previously empty fields:", original_empty_count)
    print(
        "Empty masks preserved:",
        int(check["mask_rle"].str.strip().eq("").sum())
    )
    print("Raw blank fields:", int(check["mask_rle"].eq("").sum()))
    print("Null values:", int(check.isna().sum().sum()))

print("\nV19 SUBMISSION FILES REPAIRED.")
